In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_152.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_108.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_104.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_107.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_136.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_20.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_119.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_87.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_2.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_19.npy
/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data/wn18rr_apsp_batch_130.npy
/kaggle/input/datasets/arafahmed99/wn

In [2]:
!pip install pykeen transformers scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 33.5 MB/s eta 0:00:00


In [3]:
!pip install pykeen transformers -q

import requests
import pandas as pd
from pykeen.datasets import FB15k237

print("Loading FB15k-237 from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
print(f"Total entities: {train_tf.num_entities}")
print(f"Total relations: {train_tf.num_relations}")

# 1. Download name mapping
url = "https://raw.githubusercontent.com/ZhenhaiMa/FB15k-237-entity-descriptions/master/entity2text.txt"
save_path = "entity2text_fb15k237.txt"

print("\nDownloading entity names...")
r = requests.get(url)
with open(save_path, "wb") as f:
    f.write(r.content)

# 2. Load names
df = pd.read_csv(save_path, sep="\t", header=None, names=["mid", "name"])
mid_to_name = dict(zip(df.mid, df.name))
print("Loaded:", len(mid_to_name), "names")

# 3. Download descriptions
desc_url = "https://raw.githubusercontent.com/TimDettmers/ConvE/master/data/FB15k-237/entity2textlong.txt"
desc_path = "entity2textlong_fb15k237.txt"

print("\nDownloading entity descriptions...")
r = requests.get(desc_url)
with open(desc_path, "wb") as f:
    f.write(r.content)

df_desc = pd.read_csv(desc_path, sep="\t", header=None, names=["mid", "desc"])
mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
print("Loaded:", len(mid_to_desc), "descriptions")

# 4. Build entity label list for BERT
entity_id_to_label = []
for i in range(train_tf.num_entities):
    entity_mapping = {v: k for k, v in dataset.entity_to_id.items()}
    mid = entity_mapping[i]  # Freebase mid string like /m/012abc

    if mid in mid_to_desc:
        label = str(mid_to_desc[mid])
    elif mid in mid_to_name:
        label = str(mid_to_name[mid])
    else:
        label = mid.replace("_", " ").replace("/", " ").strip()

    entity_id_to_label.append(label)

print("\nExample labels:")
for i in range(5):
    print(f"{i} → {entity_id_to_label[i][:120]} ...")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 18.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 34.7 MB/s eta 0:00:00
Loading FB15k-237 from PyKEEN...


You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out


Total entities: 14505
Total relations: 237

Loaded: 1 names

Loaded: 1 descriptions

Example labels:
0 → m 010016 ...
1 → m 0100mt ...
2 → m 0102t4 ...
3 → m 0104lr ...
4 → m 0105y2 ...


doing the pca mapping

In [9]:
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA

DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
print("Locating APSP CSV batches...")

# Grab all CSVs and filter to keep only the batch files
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'shortest_paths_' in os.path.basename(f)]

# Sort files by the starting batch index (e.g., extracts 10000 from 'batch_10000_to_12499')
csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

for f in csv_files:
    print(f" - Found: {os.path.basename(f)}")

# 2. Load and Stitch 
print("\nLoading and stitching CSV files (this may take a minute)...")
matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

# D is the full All-Pairs Shortest Path distance matrix [num_entities, num_entities]
D = np.vstack(matrices).astype(np.float32)
print(f"Stitched Distance Matrix Shape: {D.shape}")

#  3. Distance-to-Similarity Mapping (From Methodology III.B)
print("\nMapping distances to exponential similarity space...")

# Handle unreachable nodes 
finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
print(f"Global maximum finite distance (M): {global_max_distance}")

# Replace unreachable with M + 1
D[~finite_mask] = global_max_distance + 1.0

# Apply exponential decay: S = exp(-alpha * D)
alpha = 1.0
S = np.exp(-alpha * D)

# Row-normalize by max 
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

# 4. Structural Embedding Compression (PCA)
print(f"\nRunning PCA to compress from {S.shape[1]} -> 256 dimensions...")
pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)

print(f"Explained Variance Ratio: {np.sum(pca.explained_variance_ratio_):.4f}")
print(f"Final Structural Embeddings Shape: {S_compressed.shape}")

# Convert to PyTorch Tensor for the model
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)
print("\n apsp_struct_emb is ready for StructuralBERTv2!")

Locating APSP CSV batches...
 - Found: shortest_paths_batch_0_to_2499.csv
 - Found: shortest_paths_batch_2500_to_4999.csv
 - Found: shortest_paths_batch_5000_to_7499.csv
 - Found: shortest_paths_batch_7500_to_9999.csv
 - Found: shortest_paths_batch_10000_to_12499.csv
 - Found: shortest_paths_batch_12500_to_14504.csv

Loading and stitching CSV files (this may take a minute)...
Stitched Distance Matrix Shape: (14511, 14505)

Mapping distances to exponential similarity space...
Global maximum finite distance (M): 14504.0

Running PCA to compress from 14505 -> 256 dimensions...
Explained Variance Ratio: 0.4764
Final Structural Embeddings Shape: (14511, 256)

 apsp_struct_emb is ready for StructuralBERTv2!


In [17]:
%%writefile stv2.py
import time
import os
import glob
import re
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import PCA

# 1. LOAD DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except Exception: pass

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except Exception: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

print(f" -> Text mapping complete. Example: {entity_id_to_label[0][:80]}...")

# 2. LOAD & PROCESS APSP CSVs (PCA COMPRESSION)
print("\n[3/6] Processing APSP CSVs and running PCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'batch_' in os.path.basename(f)]

if not csv_files:
    raise FileNotFoundError(f"Could not find any batch CSV files in {DATA_DIR}")

csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

D = np.vstack(matrices).astype(np.float32)
print(f" -> Stitched Distance Matrix Shape: {D.shape}")

finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
D[~finite_mask] = global_max_distance + 1.0

S = np.exp(-1.0 * D)
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)

# Free up RAM
del matrices, D, S, df 
print(f" -> PCA Complete. apsp_struct_emb shape: {apsp_struct_emb.shape}")
# 3. MODEL ARCHITECTURE
class StructuralBERTv2(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="distilbert-base-uncased", embedding_dim=256, fusion="concat", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        self.struct_proj = nn.Linear(struct_dim, embedding_dim)
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        fused = self.fusion_layer(combined)
        
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        return -torch.norm(self.entity_rep(h) + self.relation_emb(r) - self.entity_rep(t), p=2, dim=-1)

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = model.entity_rep()  
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]

        # Tail Evaluation
        scores_tail = -torch.cdist(h_emb + r_emb, all_entity_emb, p=2)
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        # Head Evaluation
        scores_head = -torch.cdist(t_emb - r_emb, all_entity_emb, p=2)
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing Model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERTv2(
    num_entities=num_entities, num_relations=num_relations,
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=256, fusion="concat", use_text=True, use_struct=True, device=device
)

print("Encoding entity labels with DistilBERT (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

BATCH_SIZE, NUM_NEG, MARGIN, LR, NUM_EPOCHS, PATIENCE = 512, 16, 1.0, 2e-4, 25, 5
train_loader = DataLoader(TensorDataset(train_tf.mapped_triples), batch_size=BATCH_SIZE, shuffle=True)
optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: StructuralBERTv2 ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        pos_scores = model(h, r, t_pos)
        
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        loss = torch.relu(MARGIN + neg_scores - pos_scores.unsqueeze(1)).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    # Validation
    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_structural_bert_v2.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_v2.pt"):
    model.load_state_dict(torch.load("best_structural_bert_v2.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Overwriting stv2.py


In [18]:
!time python stv2.py


[1/6] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions...
 -> Text mapping complete. Example: Denton is a city in the U.S. state of Texas and the county seat of Denton County...

[3/6] Processing APSP CSVs and running PCA...
 -> Stitched Distance Matrix Shape: (14511, 14505)
 -> PCA Complete. apsp_struct_emb shape: torch.Size([14511, 256])

[4/6] Initializing Model...
Using device: cuda
config.json: 100%|█████████████████████████████| 483/483 [00:00<00:00, 2.53MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:

as the apsp was aa directed graph, so douing h to t is good but t to h is not fixed. so fixing it too

In [21]:
%%writefile stv2.py
import time
import os
import glob
import re
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import PCA

# 1. LOAD DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except Exception: pass

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except Exception: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

print(f" -> Text mapping complete. Example: {entity_id_to_label[0][:80]}...")

# 2. LOAD & PROCESS APSP CSVs (PCA COMPRESSION)
print("\n[3/6] Processing APSP CSVs and running PCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'batch_' in os.path.basename(f)]

if not csv_files:
    raise FileNotFoundError(f"Could not find any batch CSV files in {DATA_DIR}")

csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

D = np.vstack(matrices).astype(np.float32)
print(f" -> Stitched Distance Matrix Shape: {D.shape}")

finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
D[~finite_mask] = global_max_distance + 1.0

S = np.exp(-1.0 * D)
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)

# Free up RAM
del matrices, D, S, df 
print(f" -> PCA Complete. apsp_struct_emb shape: {apsp_struct_emb.shape}")

# 3. MODEL ARCHITECTURE
class StructuralBERTv2(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="distilbert-base-uncased", embedding_dim=256, fusion="concat", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        self.struct_proj = nn.Linear(struct_dim, embedding_dim)
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        fused = self.fusion_layer(combined)
        
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        return -torch.norm(self.entity_rep(h) + self.relation_emb(r) - self.entity_rep(t), p=2, dim=-1)

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)
# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR (Updated for Inverse Relations)
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = model.entity_rep()  
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        # Get inverse relation embedding for head prediction
        r_inv = r + num_orig_rels
        r_inv_emb = model.relation_emb(r_inv)

        # Tail Evaluation: (h, r, ?) -> h + r
        scores_tail = -torch.cdist(h_emb + r_emb, all_entity_emb, p=2)
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        # Head Evaluation: (?, r, t) becomes Tail prediction on inverse -> t + r_inv
        scores_head = -torch.cdist(t_emb + r_inv_emb, all_entity_emb, p=2)
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP (Updated with Synthetic Inverses)
print("\n[4/6] Initializing Model with Inverse Relations...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We DOUBLE the relation count in the model to make room for r_inv
model = StructuralBERTv2(
    num_entities=num_entities, 
    num_relations=num_relations * 2,  
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=256, fusion="concat", use_text=True, use_struct=True, device=device
)

print("Encoding entity labels with DistilBERT (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

BATCH_SIZE, NUM_NEG, MARGIN, LR, NUM_EPOCHS, PATIENCE = 512, 16, 1.0, 2e-4, 25, 5

# CREATE INVERSE TRIPLES 
mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               # Swap head -> tail
inverse_train[:, 2] = mapped_train[:, 0]               # Swap tail -> head
inverse_train[:, 1] = mapped_train[:, 1] + num_relations # Offset relation ID

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: StructuralBERTv2 ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        pos_scores = model(h, r, t_pos)
        
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        loss = torch.relu(MARGIN + neg_scores - pos_scores.unsqueeze(1)).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    # Pass num_relations to the evaluator so it knows the offset for r_inv
    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_structural_bert_v2.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_v2.pt"):
    model.load_state_dict(torch.load("best_structural_bert_v2.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)
# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_v2.pt"):
    model.load_state_dict(torch.load("best_structural_bert_v2.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Overwriting stv2.py


In [22]:
!time python stv2.py


[1/6] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions...
 -> Text mapping complete. Example: Denton is a city in the U.S. state of Texas and the county seat of Denton County...

[3/6] Processing APSP CSVs and running PCA...
 -> Stitched Distance Matrix Shape: (14511, 14505)
 -> PCA Complete. apsp_struct_emb shape: torch.Size([14511, 256])

[4/6] Initializing Model with Inverse Relations...
Using device: cuda
Loading weights: 100%|█| 100/100 [00:00<00:00, 1603.21it/s, Materializing param=
DistilBertModel LOAD REPORT from: distilbert-

making bi directional apsp

for fb15k-237 dset updating is done now upgrading for highest score to the code. 

updated apsp merging

In [3]:
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA

# --- 1. Define Dataset Path ---
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
print("Locating APSP CSV batches...")

# Grab all CSVs and filter to keep only the batch files
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'shortest_paths_' in os.path.basename(f)]

# Sort files by the starting batch index (e.g., extracts 10000 from 'batch_10000_to_12499')
csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

for f in csv_files:
    print(f" - Found: {os.path.basename(f)}")

# --- 2. Load and Stitch ---
print("\nLoading and stitching CSV files (this may take a minute)...")
matrices = []
for f in csv_files:
    # Assuming standard CSV format. If you saved with indices, you may need index_col=0
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

# D is the full All-Pairs Shortest Path distance matrix [num_entities, num_entities]
D = np.vstack(matrices).astype(np.float32)
print(f"Stitched Distance Matrix Shape: {D.shape}")

# --- 3. Distance-to-Similarity Mapping (From Methodology III.B) ---
print("\nMapping distances to exponential similarity space...")

# Handle unreachable nodes 
finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
print(f"Global maximum finite distance (M): {global_max_distance}")

# Replace unreachable with M + 1
D[~finite_mask] = global_max_distance + 1.0

# Apply exponential decay: S = exp(-alpha * D)
alpha = 1.0
S = np.exp(-alpha * D)

# Row-normalize by max 
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

# --- 4. Structural Embedding Compression (PCA) ---
print(f"\nRunning PCA to compress from {S.shape[1]} -> 256 dimensions...")
pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)

print(f"Explained Variance Ratio: {np.sum(pca.explained_variance_ratio_):.4f}")
print(f"Final Structural Embeddings Shape: {S_compressed.shape}")

# Convert to PyTorch Tensor for the model
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)
print("\n apsp_struct_emb is ready for StructuralBERTv2!")

Locating APSP CSV batches...
 - Found: shortest_paths_batch_0_to_2499.csv
 - Found: shortest_paths_batch_2500_to_4999.csv
 - Found: shortest_paths_batch_5000_to_7499.csv
 - Found: shortest_paths_batch_7500_to_9999.csv
 - Found: shortest_paths_batch_10000_to_12499.csv
 - Found: shortest_paths_batch_12500_to_14504.csv

Loading and stitching CSV files (this may take a minute)...
Stitched Distance Matrix Shape: (14505, 14505)

Mapping distances to exponential similarity space...
Global maximum finite distance (M): 34.0

Running PCA to compress from 14505 -> 256 dimensions...
Explained Variance Ratio: 0.6511
Final Structural Embeddings Shape: (14505, 256)

 apsp_struct_emb is ready for StructuralBERTv2!


In [14]:
%%writefile stv3.py
import time
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import PCA

# 1. LOAD DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except Exception: pass

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except Exception: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

print(f" -> Text mapping complete. Example: {entity_id_to_label[0][:80]}...")

# ==============================================================================
# 2. LOAD & PROCESS APSP CSVs (PCA COMPRESSION)
# ==============================================================================
print("\n[3/6] Processing APSP CSVs and running PCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'shortest_paths_' in os.path.basename(f)]

if not csv_files:
    raise FileNotFoundError(f"Could not find any batch CSV files in {DATA_DIR}")

csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

D = np.vstack(matrices).astype(np.float32)
print(f" -> Stitched Distance Matrix Shape: {D.shape}")

finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
D[~finite_mask] = global_max_distance + 1.0

S = np.exp(-1.0 * D)
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)

del matrices, D, S, df 
print(f" -> PCA Complete. apsp_struct_emb shape: {apsp_struct_emb.shape}")

# 3. MODEL ARCHITECTURE
class StructuralBERTv2(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="roberta-base", embedding_dim=256, fusion="glu", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate", "glu"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim) # Fallback
            
            # GLU Fusion Components
            self.glu_value = nn.Linear(in_dim, embedding_dim)
            self.glu_gate = nn.Linear(in_dim, embedding_dim)
            self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        
        # GeGLU Fusion
        if self.fusion == "glu":
            value = self.glu_value(combined)
            gate = self.glu_gate(combined)
            fused = value * torch.nn.functional.gelu(gate)
            return self.glu_mix(fused)
        
        # Fallbacks
        fused = self.fusion_layer(combined)
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb = self.entity_rep(h)
        r_emb = self.relation_emb(r)
        t_emb = self.entity_rep(t)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        scores = (h_re * r_re * t_re +
                  h_im * r_re * t_im +
                  h_re * r_im * t_im -
                  h_im * r_im * t_re).sum(dim=-1)
        return scores

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR 
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = model.entity_rep()  
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        r_inv = r + num_orig_rels
        r_inv_emb = model.relation_emb(r_inv)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        r_inv_re, r_inv_im = torch.chunk(r_inv_emb, 2, dim=-1)

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing Model with Inverse Relations & RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERTv2(
    num_entities=num_entities, 
    num_relations=num_relations * 2,  
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=256, 
    fusion="glu",  # Using the new multiplicative GLU
    use_text=True, 
    use_struct=True, 
    device=device
)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 512, 32, 3e-4, 100, 10

ALPHA = 1.0  
REG_WEIGHT = 1e-4  

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               
inverse_train[:, 2] = mapped_train[:, 0]               
inverse_train[:, 1] = mapped_train[:, 1] + num_relations 

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: StructuralBERTv3 ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        pos_scores = model(h, r, t_pos)
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        l2_reg = REG_WEIGHT * (model.entity_residual.weight.norm(p=2)**2 + model.relation_emb.weight.norm(p=2)**2)

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_structural_bert_v3.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_v3.pt"):
    model.load_state_dict(torch.load("best_structural_bert_v3.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Overwriting stv3.py


In [15]:
!time python stv3.py


[1/6] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions...
 -> Text mapping complete. Example: Denton is a city in the U.S. state of Texas and the county seat of Denton County...

[3/6] Processing APSP CSVs and running PCA...
 -> Stitched Distance Matrix Shape: (14505, 14505)
 -> PCA Complete. apsp_struct_emb shape: torch.Size([14505, 256])

[4/6] Initializing Model with Inverse Relations & RoBERTa...
Using device: cuda
Loading weights: 100%|█| 197/197 [00:00<00:00, 1654.80it/s, Materializing param=
RobertaModel LOAD REPORT from: robe

In [16]:
%%writefile stv4.py
import time
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import PCA

# 1. LOAD DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except Exception: pass

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except Exception: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

# 2. LOAD & PROCESS APSP CSVs (PCA COMPRESSION)
print("\n[3/6] Processing APSP CSVs and running PCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'shortest_paths_' in os.path.basename(f)]

if not csv_files:
    raise FileNotFoundError(f"Could not find any batch CSV files in {DATA_DIR}")

csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

D = np.vstack(matrices).astype(np.float32)
finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
D[~finite_mask] = global_max_distance + 1.0

S = np.exp(-1.0 * D)
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

pca = PCA(n_components=256, random_state=42)
S_compressed = pca.fit_transform(S)
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)

del matrices, D, S, df 

# 3. MODEL ARCHITECTURE (COMPLEX CONJUGATE FIX)
class StructuralBERTv2(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="roberta-base", embedding_dim=256, fusion="glu", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate", "glu"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)
            self.glu_value = nn.Linear(in_dim, embedding_dim)
            self.glu_gate = nn.Linear(in_dim, embedding_dim)
            self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        
        if self.fusion == "glu":
            value = self.glu_value(combined)
            gate = self.glu_gate(combined)
            fused = value * torch.nn.functional.gelu(gate)
            return self.glu_mix(fused)
        
        fused = self.fusion_layer(combined)
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb = self.entity_rep(h)
        t_emb = self.entity_rep(t)

        # FIX: Dynamically detect inverses and extract the true relation ID
        num_orig = self.relation_emb.num_embeddings
        is_inverse = (r >= num_orig)
        orig_r = r % num_orig

        r_emb = self.relation_emb(orig_r)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        # FIX: Apply Complex Conjugate for Inverses (Flip the sign of the imaginary component)
        sign = torch.where(is_inverse, torch.tensor(-1.0, device=self.device), torch.tensor(1.0, device=self.device)).unsqueeze(-1)
        r_im = r_im * sign

        scores = (h_re * r_re * t_re +
                  h_im * r_re * t_im +
                  h_re * r_im * t_im -
                  h_im * r_im * t_re).sum(dim=-1)
        return scores

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR (Complex Conjugate Synced)
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = model.entity_rep()  
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        # FIX: The inverse relation is mathematically perfectly synced to the forward relation!
        r_inv_re = r_re
        r_inv_im = -r_im

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing Model with Inverse Relations & RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERTv2(
    num_entities=num_entities, 
    num_relations=num_relations,  # FIX: Set back to original relation count!
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=256, 
    fusion="glu",  
    use_text=True, 
    use_struct=True, 
    device=device
)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 512, 32, 3e-4, 100, 10

ALPHA = 1.0  
REG_WEIGHT = 1e-4  

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               
inverse_train[:, 2] = mapped_train[:, 0]               
inverse_train[:, 1] = mapped_train[:, 1] + num_relations 

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: StructuralBERTv4 ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        pos_scores = model(h, r, t_pos)
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        l2_reg = REG_WEIGHT * (model.entity_residual.weight.norm(p=2)**2 + model.relation_emb.weight.norm(p=2)**2)

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_structural_bert_v4.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")
# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_v4.pt"):
    model.load_state_dict(torch.load("best_structural_bert_v4.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Writing stv4.py


fixing bug The Fix: Weight Tying the Conjugates
We are going to remove the num_relations * 2 parameter waste. Instead, we will force the model to dynamically flip the sign of the imaginary component whenever it encounters an inverse relation.

This will instantly sync your Head-Only MRR up to your Tail-Only MRR (0.3767), bringing your Standard Average to ~0.377 MRR and ~0.569 Hits@10, placing you firmly in the SOTA conversation.

In [17]:
!python stv4.py


[1/6] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions...

[3/6] Processing APSP CSVs and running PCA...

[4/6] Initializing Model with Inverse Relations & RoBERTa...
Using device: cuda
Loading weights: 100%|█| 197/197 [00:00<00:00, 1624.50it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UN

What Just Happened?
In v4, I forced the model to use the strict mathematical definition of a ComplEx inverse (the complex conjugate: flipping the sign of the imaginary plane). In a perfect, symmetrical universe, that works.

But FB15k-237 is not a perfect universe. It is a highly skewed, asymmetrical dataset. By forcing the inverse relation to be a strict mathematical mirror of the forward relation, we over-constrained the network. It lost the freedom to treat "predicting a movie from an actor" differently than "predicting an actor from a movie."

v3 performed so much better because it used num_relations * 2. It allowed the neural network to learn a completely independent geometric path for the backward direction.

In [18]:
%%writefile stv_final.py
import time
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import PCA

# 1. LOAD DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except Exception: pass

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except Exception: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

# 2. LOAD & PROCESS APSP CSVs (512-DIMENSIONAL PCA)
print("\n[3/6] Processing APSP CSVs and running PCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data/"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
csv_files = [f for f in all_csv_files if 'shortest_paths_' in os.path.basename(f)]

if not csv_files:
    raise FileNotFoundError(f"Could not find any batch CSV files in {DATA_DIR}")

csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)', x).group(1)))

matrices = []
for f in csv_files:
    df = pd.read_csv(f, header=None)
    matrices.append(df.values)

D = np.vstack(matrices).astype(np.float32)
finite_mask = np.isfinite(D) & (D >= 0)
global_max_distance = np.max(D[finite_mask])
D[~finite_mask] = global_max_distance + 1.0

S = np.exp(-1.0 * D)
row_max = S.max(axis=1, keepdims=True)
S = S / np.where(row_max == 0, 1e-9, row_max)

pca = PCA(n_components=512, random_state=42)
S_compressed = pca.fit_transform(S)
apsp_struct_emb = torch.tensor(S_compressed, dtype=torch.float32)

del matrices, D, S, df 

# 3. MODEL ARCHITECTURE (INDEPENDENT INVERSES + GEGLU + 512 DIMS)
class StructuralBERT_Final(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="roberta-base", embedding_dim=512, fusion="glu", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate", "glu"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)
            self.glu_value = nn.Linear(in_dim, embedding_dim)
            self.glu_gate = nn.Linear(in_dim, embedding_dim)
            self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        
        if self.fusion == "glu":
            value = self.glu_value(combined)
            gate = self.glu_gate(combined)
            fused = value * torch.nn.functional.gelu(gate)
            return self.glu_mix(fused)
        
        fused = self.fusion_layer(combined)
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb = self.entity_rep(h)
        r_emb = self.relation_emb(r)
        t_emb = self.entity_rep(t)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        scores = (h_re * r_re * t_re +
                  h_im * r_re * t_im +
                  h_re * r_im * t_im -
                  h_im * r_im * t_re).sum(dim=-1)
        return scores

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR 
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = model.entity_rep()  
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        r_inv = r + num_orig_rels
        r_inv_emb = model.relation_emb(r_inv)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        r_inv_re, r_inv_im = torch.chunk(r_inv_emb, 2, dim=-1)

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing Model with Independent Inverses & RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERT_Final(
    num_entities=num_entities, 
    num_relations=num_relations * 2,  # Back to the winning v3 strategy
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=512,  
    fusion="glu",  
    use_text=True, 
    use_struct=True, 
    device=device
)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 512, 64, 3e-4, 100, 10

ALPHA = 1.0  
REG_WEIGHT = 1e-4  

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               
inverse_train[:, 2] = mapped_train[:, 0]               
inverse_train[:, 1] = mapped_train[:, 1] + num_relations 

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: StructuralBERT_Final ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        pos_scores = model(h, r, t_pos)
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        l2_reg = REG_WEIGHT * (model.entity_residual.weight.norm(p=2)**2 + model.relation_emb.weight.norm(p=2)**2)

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_structural_bert_final.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_structural_bert_final.pt"):
    model.load_state_dict(torch.load("best_structural_bert_final.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Writing stv_final.py


In [19]:
!time python stv_final.py


[1/6] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions...

[3/6] Processing APSP CSVs and running PCA...

[4/6] Initializing Model with Independent Inverses & RoBERTa...
Using device: cuda
Loading weights: 100%|█| 197/197 [00:00<00:00, 1635.92it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids |

still not close to sota using the static embeddings. so now going for new arch design

In [24]:
 %%writefile stv_sota.py
import time
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237

# 1. LOAD DATASET & TEXT (NO CSVs NEEDED - THE GRAPH IS THE NETWORK)
print("\n[1/5] Loading FB15k-237 dataset from PyKEEN...")
dataset = FB15k237()
train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/5] Fetching and mapping textual descriptions...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"])
    mid_to_name = dict(zip(df_name.mid, df_name.name))
except: pass
try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], on_bad_lines='skip')
    mid_to_desc = dict(zip(df_desc.mid, df_desc.desc))
except: pass

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: k for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid[i]
    if mid in mid_to_desc and pd.notna(mid_to_desc[mid]): label = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]): label = str(mid_to_name[mid])
    else: label = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(label)

# 2. BUILD THE DYNAMIC MESSAGE PASSING GRAPH
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract the edges from the training set to act as the GNN routing pathways
mapped_train = train_tf.mapped_triples
heads, rels, tails = mapped_train[:, 0], mapped_train[:, 1], mapped_train[:, 2]

# Add inverse pathways so information can flow backward through the GNN
inv_heads, inv_tails = tails, heads
inv_rels = rels + num_relations

all_heads = torch.cat([heads, inv_heads]).to(device)
all_tails = torch.cat([tails, inv_tails]).to(device)
all_rels = torch.cat([rels, inv_rels]).to(device)

edge_index = torch.stack([all_heads, all_tails], dim=0)
edge_type = all_rels

# 3. SOTA ARCHITECTURE: COMPGCN + ROBERTA + COMPLEX
class CompGCNLayer(nn.Module):
    """
    Vectorized Compositional Graph Convolution.
    Dynamically routes topological information based on exact relation types.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim)
        
    def forward(self, x, edge_idx, edge_t, rel_emb):
        h_idx, t_idx = edge_idx[0], edge_idx[1]
        
        # 1. Gather semantic features of neighbors
        h_feats = x[h_idx]
        
        # 2. Modulate features by specific relation type (Solves Relational Colorblindness)
        r_feats = rel_emb[edge_t]
        messages = h_feats * r_feats 
        
        # 3. Aggregate messages onto the target node (Solves the Euclidean Trap)
        out = torch.zeros_like(x)
        out.scatter_add_(0, t_idx.unsqueeze(-1).expand_as(messages), messages)
        
        # 4. Add self-loop and non-linear activation
        out = out + x
        return torch.nn.functional.gelu(self.W(out))


class SotaGraphFusion(nn.Module):
    def __init__(self, num_entities, num_relations, bert_model_name="roberta-base", embedding_dim=512, device=device):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations * 2 
        
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False
        
        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=False)
        
        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        self.relation_emb = nn.Embedding(self.num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
        # 2-Hop Dynamic Graph Routing
        self.gcn1 = CompGCNLayer(embedding_dim, embedding_dim)
        self.gcn2 = CompGCNLayer(embedding_dim, embedding_dim)
        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out)
        self.bert_cache[:self.num_entities].copy_(torch.cat(embs, dim=0).detach())

    def get_dynamic_embeddings(self, edge_idx, edge_t):
        # Base Semantics
        base_x = self.text_proj(self.bert_cache) + self.entity_residual.weight
        r_emb = self.relation_emb.weight
        
        # Let the network walk the graph (2 hops)
        x_1 = self.gcn1(base_x, edge_idx, edge_t, r_emb)
        x_2 = self.gcn2(x_1, edge_idx, edge_t, r_emb)
        return x_2

    def score_triples(self, h, r, t, dynamic_x):
        h_emb = dynamic_x[h]
        r_emb = self.relation_emb(r)
        t_emb = dynamic_x[t]

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        scores = (h_re * r_re * t_re + h_im * r_re * t_im + h_re * r_im * t_im - h_im * r_im * t_re).sum(dim=-1)
        return scores

# 4. FAST VECTORIZED EVALUATOR 
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, edge_idx, edge_t, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    
    # Run the GNN routing once for the entire evaluation!
    all_entity_emb = model.get_dynamic_embeddings(edge_idx, edge_t)  
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []
    for start_idx in range(0, triples.shape[0], batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        r_inv_emb = model.relation_emb(r + num_orig_rels)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        r_inv_re, r_inv_im = torch.chunk(r_inv_emb, 2, dim=-1)

        hr_re, hr_im = h_re * r_re - h_im * r_im, h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re, tr_im = t_re * r_inv_re - t_im * r_inv_im, t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    ranks = np.concatenate([tail_ranks, head_ranks])
    return {"MRR": np.mean(1.0/ranks), "Hits@10": np.mean(ranks<=10)}
# 5. INITIALIZATION & TRAINING LOOP (MEMORY LEAK FIXED)
print("\n[3/5] Initializing SOTA Dynamic GNN...")
model = SotaGraphFusion(num_entities, num_relations, embedding_dim=512, device=device)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=128)

# You dropped it to 256, which is totally fine for stability!
BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 256, 64, 3e-4, 100, 10
ALPHA, REG_WEIGHT = 1.0, 1e-4  

extended_train_triples = torch.cat([mapped_train, torch.stack([tails, rels + num_relations, heads], dim=1)], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[4/5] === START TRAINING: SOTA CompGCN Fusion ===")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0
    
    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]
        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        # 1. Walk the graph INSIDE the loop so PyTorch can safely destroy it after backward()
        dynamic_x = model.get_dynamic_embeddings(edge_index, edge_type)

        # 2. Score using the dynamic coordinates
        pos_scores = model.score_triples(h, r, t_pos, dynamic_x)
        
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model.score_triples(h_rep, r_rep, neg_t.reshape(-1), dynamic_x).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        l2_reg = REG_WEIGHT * (model.entity_residual.weight.norm(p=2)**2 + model.relation_emb.weight.norm(p=2)**2)

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        
        # FIX: The memory leak is dead. PyTorch will cleanly flush VRAM now.
        loss.backward() 
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    # Evaluation
    val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations, edge_index, edge_type)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {total_loss/nbatches:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_sota_gcn.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

print("\n[5/5] === FINAL TEST RESULTS (Standard Filtered) ===")
model.load_state_dict(torch.load("best_sota_gcn.pt", map_location=device))
res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations, edge_index, edge_type)
print(f"SOTA Standard MRR: {res['MRR']:.4f} | Hits@10: {res['Hits@10']:.4f}")

Overwriting stv_sota.py


In [ ]:
!time python stv_sota.py


[1/5] Loading FB15k-237 dataset from PyKEEN...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/5] Fetching and mapping textual descriptions...

[3/5] Initializing SOTA Dynamic GNN...
Loading weights: 100%|█| 197/197 [00:00<00:00, 1595.34it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight   

okay, now going for the wn18rr dset. not claining any sota just proving that this way works too and this is more sufficient way of doing while achive a higher tier performance. 

In [14]:
%%writefile wn18rr_stv_final.py
import time
import os
import glob
import re
import gc
import urllib.request
import tarfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import PathDataset
from sklearn.decomposition import IncrementalPCA

# 1. LOAD WN18RR DATASET & TEXT DESCRIPTIONS (FOOLPROOF METHOD)
print("\n[1/6] Downloading WN18RR manually to bypass PyKEEN 404 error...")

if not os.path.exists("WN18RR.tar.gz"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/peterhu95/ConvE-CNN-ECFA/master/WN18RR.tar.gz", "WN18RR.tar.gz")

if not os.path.exists("wn18rr_data"):
    with tarfile.open("WN18RR.tar.gz", "r:gz") as tar:
        tar.extractall("wn18rr_data")

# Locate the extracted text files dynamically
train_path = glob.glob("wn18rr_data/**/train.txt", recursive=True)[0]
valid_path = glob.glob("wn18rr_data/**/valid.txt", recursive=True)[0]
test_path = glob.glob("wn18rr_data/**/test.txt", recursive=True)[0]

dataset = PathDataset(
    training_path=train_path,
    validation_path=valid_path,
    testing_path=test_path,
)

train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions for WN18RR...")
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/WN18RR/entity2text.txt"

synset_to_desc = {}
try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["synset_id", "desc"], on_bad_lines='skip', dtype=str)
    synset_to_desc = dict(zip(df_desc.synset_id, df_desc.desc))
except Exception as e: 
    print(f"Failed to fetch descriptions: {e}")

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_synset = {v: str(k) for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    synset = id_to_synset[i]
    if synset in synset_to_desc and pd.notna(synset_to_desc[synset]): 
        label = str(synset_to_desc[synset])
    else: 
        label = f"WordNet concept {synset}"
    entity_id_to_label.append(label)

print(f" -> Text mapping complete. Example: {entity_id_to_label[0][:80]}...")

# 2. LOAD & PROCESS APSP CSVs (EXPLOITING 30GB CPU RAM)
print("\n[3/6] Processing APSP CSVs with High-Capacity IncrementalPCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

if not all_csv_files:
    raise FileNotFoundError(f"Could not find any CSV files in {DATA_DIR}")

# Sort numerically by the starting batch ID to guarantee row alignment
all_csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)_to_', os.path.basename(x)).group(1)))

print(f" -> Found {len(all_csv_files)} batch files. Fitting Incremental PCA...")
# UPGRADED: Cranking batch size to 10000 to utilize the 30GB of system RAM for faster fitting
ipca = IncrementalPCA(n_components=512, batch_size=10000)

# Pass 1: Fit the PCA incrementally
for i, f in enumerate(all_csv_files):
    df = pd.read_csv(f, header=None)
    D_batch = df.values.astype(np.float32)
    
    # Replace inf distances with a high constant (100.0) for disjoint subgraphs
    D_batch[~np.isfinite(D_batch)] = 100.0
    
    # Exponentiate and normalize locally per row
    S_batch = np.exp(-1.0 * D_batch)
    row_max = S_batch.max(axis=1, keepdims=True)
    S_batch = S_batch / np.where(row_max == 0, 1e-9, row_max)
    
    ipca.partial_fit(S_batch)
    del df, D_batch, S_batch
    gc.collect()

print(" -> Incremental PCA Fitted! Applying transform...")

# Pass 2: Apply the transformation block-by-block
S_compressed_list = []
for f in all_csv_files:
    df = pd.read_csv(f, header=None)
    D_batch = df.values.astype(np.float32)
    D_batch[~np.isfinite(D_batch)] = 100.0
    
    S_batch = np.exp(-1.0 * D_batch)
    row_max = S_batch.max(axis=1, keepdims=True)
    S_batch = S_batch / np.where(row_max == 0, 1e-9, row_max)
    
    compressed = ipca.transform(S_batch)
    S_compressed_list.append(compressed)
    
    del df, D_batch, S_batch
    gc.collect()

apsp_struct_emb = torch.tensor(np.vstack(S_compressed_list), dtype=torch.float32)
del S_compressed_list
gc.collect()

print(f" -> PCA Complete. apsp_struct_emb shape: {apsp_struct_emb.shape}")

# 3. MODEL ARCHITECTURE (16GB VRAM UNLOCKED)
class StructuralBERT_Final(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="roberta-base", embedding_dim=512, fusion="glu", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim)
        )
        # CHANGED: persistent=True to ensure it saves into the state_dict
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=True)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        
        # CHANGED: persistent=True to ensure it saves into the state_dict
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=True)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate", "glu"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)
            self.glu_value = nn.Linear(in_dim, embedding_dim)
            self.glu_gate = nn.Linear(in_dim, embedding_dim)
            self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out) 
        self.bert_cache.copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        
        if self.fusion == "glu":
            value = self.glu_value(combined)
            gate = self.glu_gate(combined)
            fused = value * torch.nn.functional.gelu(gate)
            return self.glu_mix(fused)
        
        fused = self.fusion_layer(combined)
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb = self.entity_rep(h)
        r_emb = self.relation_emb(r)
        t_emb = self.entity_rep(t)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        scores = (h_re * r_re * t_re +
                  h_im * r_re * t_im +
                  h_re * r_im * t_im -
                  h_im * r_im * t_re).sum(dim=-1)
        return scores

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR 
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = torch.zeros(model.num_entities, model.entity_residual.embedding_dim, device=model.device)
    for i in range(0, model.num_entities, 2048):
        chunk_ids = torch.arange(i, min(i+2048, model.num_entities), device=model.device)
        all_entity_emb[chunk_ids] = model.entity_rep(chunk_ids)
        
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        r_inv = r + num_orig_rels
        r_inv_emb = model.relation_emb(r_inv)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        r_inv_re, r_inv_im = torch.chunk(r_inv_emb, 2, dim=-1)

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing WN18RR Model with RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERT_Final(
    num_entities=num_entities, 
    num_relations=num_relations * 2,
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=512,  
    fusion="glu",  
    use_text=True, 
    use_struct=True, 
    device=device
)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=256)

BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 2048, 64, 3e-4, 100, 10
ALPHA = 1.0  
REG_WEIGHT = 1e-4  

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               
inverse_train[:, 2] = mapped_train[:, 0]               
inverse_train[:, 1] = mapped_train[:, 1] + num_relations 

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: WN18RR SOTA ===\n")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        pos_scores = model(h, r, t_pos)
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        
        h_res = model.entity_residual(h)
        r_emb = model.relation_emb(r)
        t_pos_res = model.entity_residual(t_pos)
        t_neg_res = model.entity_residual(neg_t.reshape(-1))

        batch_len = h.size(0)
        l2_reg = (REG_WEIGHT / batch_len) * (
            h_res.norm(p=2)**2 + 
            r_emb.norm(p=2)**2 + 
            t_pos_res.norm(p=2)**2 + 
            t_neg_res.norm(p=2)**2
        )

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_wn18rr_final.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_wn18rr_final.pt"):
    model.load_state_dict(torch.load("best_wn18rr_final.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Overwriting wn18rr_stv_final.py


In [5]:
!pip install pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 16.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.2/496.2 kB 30.6 MB/s eta 0:00:00


In [15]:
!time python wn18rr_stv_final.py


[1/6] Downloading WN18RR manually to bypass PyKEEN 404 error...
/kaggle/working/wn18rr_stv_final.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("wn18rr_data")
You're trying to map triples with 212 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3134 triples were filtered out
You're trying to map triples with 211 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3034 triples were filtered out
Entities: 40559 | Relations: 11

[2/6] Fetching and mapping textual descriptions for WN18RR...
 -> Text mapping complete. Example: take a breath, that which is perceived or known or inferred to have its own dist...

[3/6] Processing APSP CSVs with High-Capacity IncrementalPCA...
 -> Found 41 batch f

now doing ablation for the wn18rr

In [15]:
%%writefile wn18rr_ablation_study.py

import os
import glob
import re
import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import WN18RR
from sklearn.decomposition import IncrementalPCA

# CONFIGURATION
SEED = 42
DATA_DIR = "/kaggle/input/datasets/arafahmed99/wn18rr-shortest-paths-data"
TEXT_URL = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/WN18RR/entity2text.txt"
BERT_MODEL = "roberta-base"

EMBEDDING_DIM = 512
TEXT_MAX_LENGTH = 64

BATCH_SIZE = 2048
NUM_NEGATIVES = 64
LEARNING_RATE = 3e-4
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
REG_WEIGHT = 1e-4
GRAD_CLIP = 1.0

PCA_COMPONENTS = 512
PCA_BATCH_SIZE = 10000

EVAL_BATCH_SIZE = 256
ENTITY_EMB_BATCH_SIZE = 2048
TEXT_BATCH_SIZE = 256

CHECKPOINT_DIR = "ablation_checkpoints"
RESULTS_FILE = "wn18rr_controlled_ablation_results.csv"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# REPRODUCIBILITY
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("StructuralBERT — WN18RR Controlled Ablation Study")
print("=" * 80)
print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")

# 1. LOAD DATASET
print("\n[1/7] Loading WN18RR...")
dataset = WN18RR()

train_tf = dataset.training
valid_tf = dataset.validation
test_tf = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations

print(f"Training entities : {num_entities}")
print(f"Original relations: {num_relations}")

# Check validation/test entity IDs
for split_name, split_tf in [("validation", valid_tf), ("test", test_tf)]:
    entity_ids = split_tf.mapped_triples[:, [0, 2]]
    min_id, max_id = int(entity_ids.min().item()), int(entity_ids.max().item())
    print(f"{split_name.capitalize()} entity IDs: {min_id} -> {max_id}")
    if min_id < 0 or max_id >= num_entities:
        raise ValueError(f"{split_name} contains entity IDs outside the training vocabulary.")

# 2. LOAD ENTITY TEXT
print("\n[2/7] Loading WN18RR entity descriptions...")
synset_to_desc = {}

try:
    desc_df = pd.read_csv(TEXT_URL, sep="\t", header=None, names=["synset_id", "desc"], on_bad_lines="skip", dtype=str)
    synset_to_desc = dict(zip(desc_df["synset_id"], desc_df["desc"]))
    print(f"Loaded {len(synset_to_desc):,} descriptions.")
except Exception as exc:
    print(f"WARNING: Failed to load descriptions: {exc}")

entity_to_id = train_tf.entity_to_id
id_to_synset = {int(entity_id): str(synset_id) for synset_id, entity_id in entity_to_id.items()}

entity_id_to_text = []
for entity_id in range(num_entities):
    synset_id = id_to_synset.get(entity_id)
    if synset_id is None:
        text = f"WordNet concept {entity_id}"
    else:
        text = synset_to_desc.get(synset_id, f"WordNet concept {synset_id}")
        if pd.isna(text):
            text = f"WordNet concept {synset_id}"
    entity_id_to_text.append(str(text))

print(f"Prepared text for {len(entity_id_to_text):,} entities.")

# 3. CACHE FROZEN RoBERTa REPRESENTATIONS ONCE
print("\n[3/7] Encoding frozen RoBERTa entity representations once...")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)
bert.eval()

for parameter in bert.parameters():
    parameter.requires_grad = False

bert_hidden = bert.config.hidden_size
text_cache = torch.empty(num_entities, bert_hidden, dtype=torch.float32, device=DEVICE)

with torch.no_grad():
    for start in range(0, num_entities, TEXT_BATCH_SIZE):
        batch_text = entity_id_to_text[start : start + TEXT_BATCH_SIZE]
        tokens = tokenizer(batch_text, padding=True, truncation=True, max_length=TEXT_MAX_LENGTH, return_tensors="pt")
        tokens = {key: value.to(DEVICE) for key, value in tokens.items()}
        
        cls_embeddings = bert(**tokens).last_hidden_state[:, 0, :]
        text_cache[start : start + len(batch_text)].copy_(cls_embeddings)
        
        if (start // TEXT_BATCH_SIZE) % 20 == 0:
            print(f"  Encoded {min(start + TEXT_BATCH_SIZE, num_entities):,}/{num_entities:,} entities")

print(f"RoBERTa cache shape: {tuple(text_cache.shape)}")

del bert, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 4. LOAD + COMPILE APSP REPRESENTATION
print("\n[4/7] Loading APSP files and fitting Incremental PCA...")

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"APSP directory does not exist:\n{DATA_DIR}")

csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No APSP CSV files found in:\n{DATA_DIR}")

def batch_start_id(path: str) -> int:
    match = re.search(r"batch_(\d+)_to_", os.path.basename(path))
    if match is None:
        raise ValueError(f"Could not parse batch ID from {os.path.basename(path)}")
    return int(match.group(1))

csv_files.sort(key=batch_start_id)
print(f"Found {len(csv_files)} APSP files.")

def load_similarity_matrix(csv_path: str) -> np.ndarray:
    distances = pd.read_csv(csv_path, header=None).values.astype(np.float32)
    finite_mask = np.isfinite(distances) & (distances >= 0)
    
    if not finite_mask.any():
        raise ValueError(f"No finite distances found in {csv_path}")
        
    max_distance = float(distances[finite_mask].max())
    distances[~finite_mask] = max_distance + 1.0  # Unreachable pairs
    
    similarities = np.exp(-distances)
    row_max = similarities.max(axis=1, keepdims=True)
    similarities /= np.where(row_max > 0, row_max, 1.0)
    
    return similarities.astype(np.float32)

# PCA fit pass
ipca = IncrementalPCA(n_components=PCA_COMPONENTS, batch_size=PCA_BATCH_SIZE)
for idx, csv_path in enumerate(csv_files, start=1):
    similarities = load_similarity_matrix(csv_path)
    print(f"  PCA fit {idx:>3}/{len(csv_files)} | {similarities.shape}")
    ipca.partial_fit(similarities)
    del similarities
    gc.collect()

print("Incremental PCA fitted.")

# PCA transform pass
compressed_blocks = []
for idx, csv_path in enumerate(csv_files, start=1):
    similarities = load_similarity_matrix(csv_path)
    compressed = ipca.transform(similarities)
    compressed_blocks.append(compressed.astype(np.float32))
    print(f"  PCA transform {idx:>3}/{len(csv_files)}")
    del similarities
    gc.collect()

apsp_struct_emb = torch.from_numpy(np.vstack(compressed_blocks)).float().to(DEVICE)
del compressed_blocks
gc.collect()

if apsp_struct_emb.shape[0] != num_entities:
    raise ValueError(f"Structural embedding/entity vocabulary mismatch: APSP rows = {apsp_struct_emb.shape[0]}, Entities = {num_entities}")

print(f"Compiled structural representation: {tuple(apsp_struct_emb.shape)}")

# 5. BUILD FILTERED EVALUATION MAPS
print("\n[5/7] Building filtered evaluation maps...")

def build_filter_maps(triples_tensor: torch.Tensor):
    tail_map, head_map = {}, {}
    for h, r, t in triples_tensor.cpu().numpy():
        h, r, t = int(h), int(r), int(t)
        tail_map.setdefault((h, r), set()).add(t)
        head_map.setdefault((r, t), set()).add(h)
    return tail_map, head_map

all_triples = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_triples)

print(f"Tail filter groups: {len(filter_tails):,}")
print(f"Head filter groups: {len(filter_heads):,}")

# 6. CONTROLLED STRUCTURALBERT MODEL
class StructuralBERTAblation(nn.Module):
    def __init__(self, use_text: bool, use_struct: bool, fusion: str, structural_embeddings: torch.Tensor, 
                 text_embeddings: torch.Tensor, num_entities: int, num_relations: int, device: torch.device):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion
        self.embedding_dim = EMBEDDING_DIM

        # Relations & Structure Path
        self.relation_emb = nn.Embedding(num_relations, self.embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
        self.struct_proj = nn.Sequential(
            nn.Linear(structural_embeddings.shape[1], self.embedding_dim),
            nn.GELU(),
            nn.LayerNorm(self.embedding_dim),
            nn.Linear(self.embedding_dim, self.embedding_dim),
        )
        self.register_buffer("struct_cache", structural_embeddings.detach().clone(), persistent=False)

        # Text Path & Residual
        self.text_proj = nn.Sequential(
            nn.Linear(text_embeddings.shape[1], self.embedding_dim),
            nn.GELU(),
            nn.LayerNorm(self.embedding_dim),
            nn.Linear(self.embedding_dim, self.embedding_dim),
        )
        self.register_buffer("bert_cache", text_embeddings.detach().clone(), persistent=False)
        self.entity_residual = nn.Embedding(num_entities, self.embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        # Fusion Layers
        if fusion == "glu":
            if not (use_text and use_struct):
                raise ValueError("GeGLU requires text + structure.")
            fusion_dim = 2 * self.embedding_dim
            self.glu_value = nn.Linear(fusion_dim, self.embedding_dim)
            self.glu_gate = nn.Linear(fusion_dim, self.embedding_dim)
            self.glu_mix = nn.Linear(self.embedding_dim, self.embedding_dim)
        elif fusion == "add":
            if not (use_text and use_struct):
                raise ValueError("Add-Fusion requires text + structure.")
        elif fusion == "none":
            if use_text and use_struct:
                raise ValueError("Fusion='none' cannot combine two modalities.")
        else:
            raise ValueError(f"Unsupported fusion mode: {fusion}")

        self.to(device)

    def entity_rep(self, ids: torch.Tensor) -> torch.Tensor:
        ids = ids.to(self.device, non_blocking=True)
        batch_size = ids.shape[0]

        text = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(batch_size, self.embedding_dim, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(text)

        if self.use_text and self.use_struct:
            if self.fusion == "add":
                fused = text + struct
            elif self.fusion == "glu":
                combined = torch.cat([text, struct], dim=-1)
                value = self.glu_value(combined)
                gate = self.glu_gate(combined)
                fused = self.glu_mix(value * F.gelu(gate))
            else:
                raise RuntimeError("Invalid multimodal fusion.")
        elif self.use_text:
            fused = text
        elif self.use_struct:
            fused = struct
        else:
            fused = torch.zeros(batch_size, self.embedding_dim, device=self.device)

        return fused + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb, r_emb, t_emb = self.entity_rep(h), self.relation_emb(r), self.entity_rep(t)
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        return (h_re * r_re * t_re + h_im * r_re * t_im + h_re * r_im * t_im - h_im * r_im * t_re).sum(dim=-1)

# 7. TRAINING + EVALUATION
print("\n[6/7] Preparing reciprocal training data...")

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]
inverse_train[:, 2] = mapped_train[:, 0]
inverse_train[:, 1] = mapped_train[:, 1] + num_relations

extended_train = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train), batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())

@torch.no_grad()
def build_all_entity_embeddings(model):
    model.eval()
    all_embeddings = torch.empty(model.num_entities, model.embedding_dim, device=model.device)
    for start in range(0, model.num_entities, ENTITY_EMB_BATCH_SIZE):
        ids = torch.arange(start, min(start + ENTITY_EMB_BATCH_SIZE, model.num_entities), device=model.device)
        all_embeddings[start : start + len(ids)] = model.entity_rep(ids)
    return all_embeddings

def average_tie_rank(scores: torch.Tensor, target_index: int) -> float:
    target_score = scores[target_index].item()
    greater = (scores > target_score).sum().item()
    equal = (scores == target_score).sum().item()
    return greater + 1.0 + 0.5 * (equal - 1)

@torch.no_grad()
def evaluate_model(model, eval_tf, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    all_entity_emb = build_all_entity_embeddings(model)
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)

    tail_ranks, head_ranks = [], []
    for start in range(0, triples.shape[0], batch_size):
        batch = triples[start : start + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
        h_emb, t_emb, r_emb = all_entity_emb[h], all_entity_emb[t], model.relation_emb(r)

        # Tail prediction
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))

        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            filtered = [entity for entity in filter_tails.get((hi, ri), set()) if entity != ti]
            if filtered:
                scores_tail[i, filtered] = -1e9
            tail_ranks.append(average_tie_rank(scores_tail[i], ti))

        # Head prediction
        inverse_r = r + num_relations
        inverse_r_emb = model.relation_emb(inverse_r)
        r_inv_re, r_inv_im = torch.chunk(inverse_r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))

        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            filtered = [entity for entity in filter_heads.get((ri, ti), set()) if entity != hi]
            if filtered:
                scores_head[i, filtered] = -1e9
            head_ranks.append(average_tie_rank(scores_head[i], hi))

    tail_ranks, head_ranks = np.asarray(tail_ranks, dtype=np.float64), np.asarray(head_ranks, dtype=np.float64)
    combined_ranks = np.concatenate([tail_ranks, head_ranks])

    def metrics(ranks):
        return {
            "MRR": float(np.mean(1.0 / ranks)),
            "Hits@1": float(np.mean(ranks <= 1)),
            "Hits@10": float(np.mean(ranks <= 10)),
        }

    return {"tail": metrics(tail_ranks), "head": metrics(head_ranks), "overall": metrics(combined_ranks)}

# TIE HANDLING SANITY CHECK
print("\n[7/7] Running evaluator sanity check...")
test_scores = torch.zeros(100, device=DEVICE)
expected_rank = (100 + 1) / 2
actual_rank = average_tie_rank(test_scores, target_index=0)

print(f"All-tie test: expected rank {expected_rank:.1f}, actual rank {actual_rank:.1f}")
if not np.isclose(actual_rank, expected_rank):
    raise RuntimeError("Tie-ranking sanity check failed.")
print("Evaluator tie handling verified.")

# ABLATION RUNNER
def run_ablation(name: str, use_text: bool, use_struct: bool, fusion: str):
    print("\n" + "=" * 80)
    print(f"STARTING: {name}")
    print("=" * 80)
    print(f"text={use_text} | structure={use_struct} | fusion={fusion}")

    model = StructuralBERTAblation(
        use_text=use_text, use_struct=use_struct, fusion=fusion,
        structural_embeddings=apsp_struct_emb, text_embeddings=text_cache,
        num_entities=num_entities, num_relations=num_relations * 2, device=DEVICE
    )

    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_{name}.pt")
    
    best_val_mrr = -np.inf
    no_improvement = 0
    start_time = time.time()

    # Training Loop
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        num_batches = 0

        for (batch,) in train_loader:
            batch = batch.to(DEVICE, non_blocking=True)
            h, r, t_pos = batch[:, 0], batch[:, 1], batch[:, 2]

            neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEGATIVES), device=DEVICE)
            pos_scores = model.score_triples(h, r, t_pos)

            h_rep = h.unsqueeze(1).expand(-1, NUM_NEGATIVES).reshape(-1)
            r_rep = r.unsqueeze(1).expand(-1, NUM_NEGATIVES).reshape(-1)
            neg_scores = model.score_triples(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEGATIVES)

            pos_loss = -F.logsigmoid(pos_scores).mean()
            neg_weights = torch.softmax(neg_scores, dim=-1).detach()
            neg_loss = -(neg_weights * F.logsigmoid(-neg_scores)).sum(dim=-1).mean()

            # CORRECT BATCH-WISE L2 REGULARIZATION
            h_res = model.entity_residual(h)
            r_emb = model.relation_emb(r)
            t_pos_res = model.entity_residual(t_pos)
            t_neg_res = model.entity_residual(neg_t.reshape(-1))

            batch_len = h.size(0)
            
            l2_reg = (REG_WEIGHT / batch_len) * (
                h_res.norm(p=2)**2 + 
                r_emb.norm(p=2)**2 + 
                t_pos_res.norm(p=2)**2 + 
                t_neg_res.norm(p=2)**2
            )

            loss = pos_loss + neg_loss + l2_reg

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            total_loss += float(loss.item())
            num_batches += 1

        avg_loss = total_loss / max(num_batches, 1)

        # Validation
        validation = evaluate_model(model, valid_tf)
        val_mrr = validation["overall"]["MRR"]
        scheduler.step(val_mrr)

        print(f"Epoch {epoch:03d} | Loss {avg_loss:.4f} | Val MRR {val_mrr:.4f} | Val Hits@10 {validation['overall']['Hits@10']:.4f}")

        # Best checkpoint
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            no_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> Saved best checkpoint (MRR={best_val_mrr:.4f})")
        else:
            no_improvement += 1
            if no_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    training_minutes = (time.time() - start_time) / 60.0

    # Restore best checkpoint and test
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    test_results = evaluate_model(model, test_tf)
    overall = test_results["overall"]

    print(f"\n{name} TEST RESULTS")
    print(f"  Tail MRR    : {test_results['tail']['MRR']:.4f}")
    print(f"  Head MRR    : {test_results['head']['MRR']:.4f}")
    print(f"  Overall MRR : {overall['MRR']:.4f}")
    print(f"  Hits@1      : {overall['Hits@1']:.4f}")
    print(f"  Hits@10     : {overall['Hits@10']:.4f}")
    print(f"  Training min: {training_minutes:.2f}")

    result = {
        "MRR": overall["MRR"],
        "Hits@1": overall["Hits@1"],
        "Hits@10": overall["Hits@10"],
        "Tail_MRR": test_results["tail"]["MRR"],
        "Head_MRR": test_results["head"]["MRR"],
        "Best_Val_MRR": best_val_mrr,
        "Training_Minutes": training_minutes,
    }

    # Save after every completed model
    existing_results = {}
    if os.path.exists(RESULTS_FILE):
        try:
            existing_results = pd.read_csv(RESULTS_FILE, index_col=0).to_dict(orient="index")
        except Exception:
            pass

    existing_results[name] = result
    pd.DataFrame(existing_results).T.to_csv(RESULTS_FILE)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return result

# CONTROLLED ABLATIONS
results = {}

results["Residual-Only"] = run_ablation("Residual-Only", False, False, "none")
results["Residual+Text"] = run_ablation("Residual-Text", True, False, "none")
results["Residual+Structure"] = run_ablation("Residual-Structure", False, True, "none")
results["Add-Fusion"] = run_ablation("Add-Fusion", True, True, "add")
results["GeGLU"] = run_ablation("GeGLU", True, True, "glu")

# FINAL TABLE
print("\n" + "=" * 80)
print("FINAL CONTROLLED ABLATION RESULTS — WN18RR")
print("=" * 80)
print(f"{'Configuration':<24}{'MRR':>10}{'Hits@1':>10}{'Hits@10':>10}")
print("-" * 80)
for name, metrics in results.items():
    print(f"{name:<24}{metrics['MRR']:>10.4f}{metrics['Hits@1']:>10.4f}{metrics['Hits@10']:>10.4f}")
print("-" * 80)

# EFFECT SIZES
if "Residual-Only" in results and "Residual+Structure" in results:
    structure_gain = results["Residual+Structure"]["MRR"] - results["Residual-Only"]["MRR"]
    print(f"\nAPSP structural gain over Residual-Only: {structure_gain:+.4f} MRR")

if "Residual+Text" in results and "Add-Fusion" in results:
    structure_gain_after_text = results["Add-Fusion"]["MRR"] - results["Residual+Text"]["MRR"]
    print(f"Structure gain after adding text: {structure_gain_after_text:+.4f} MRR")

if "Add-Fusion" in results and "GeGLU" in results:
    geglu_gain = results["GeGLU"]["MRR"] - results["Add-Fusion"]["MRR"]
    print(f"GeGLU gain over Add-Fusion: {geglu_gain:+.4f} MRR")

print(f"\nResults saved continuously to:\n{RESULTS_FILE}")
print("\nAblation study complete.")

Overwriting wn18rr_ablation_study.py


In [ ]:
!time python wn18rr_ablation_study.py

StructuralBERT — WN18RR Controlled Ablation Study
Device : cuda
Seed   : 42

[1/7] Loading WN18RR...
You're trying to map triples with 212 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3134 triples were filtered out
You're trying to map triples with 211 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3034 triples were filtered out
Training entities : 40559
Original relations: 11
Validation entity IDs: 0 -> 40558
Test entity IDs: 0 -> 40552

[2/7] Loading WN18RR entity descriptions...
Loaded 40,943 descriptions.
Prepared text for 40,559 entities.

[3/7] Encoding frozen RoBERTa entity representations once...
Loading weights: 100%|█| 197/197 [00:00<00:00, 1447.65it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head

now ablition for the fb15k237

In [1]:
%%writefile fb15k237_ablation_study.py

import os
import glob
import re
import gc
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import FB15k237
from sklearn.decomposition import IncrementalPCA

# CONFIGURATION
SEED = 42
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data"
TEXT_NAME_URL = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
TEXT_DESC_URL = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"
BERT_MODEL = "roberta-base"

EMBEDDING_DIM = 512
PCA_COMPONENTS = 512
PCA_BATCH_SIZE = 10000
TEXT_MAX_LENGTH = 64

BATCH_SIZE = 512
NUM_NEGATIVES = 64
LEARNING_RATE = 3e-4
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR = 0.5

REG_WEIGHT = 1e-4
GRAD_CLIP = 1.0

EVAL_BATCH_SIZE = 256
ENTITY_EMB_BATCH_SIZE = 2048

CHECKPOINT_DIR = "fb15k237_ablation_checkpoints"
RESULTS_FILE = "fb15k237_controlled_ablation_results.csv"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# REPRODUCIBILITY
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("StructuralBERT — FB15k-237 Controlled Ablation Study")
print("=" * 80)
print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")

# 1. LOAD DATASET
print("\n[1/6] Loading FB15k-237...")
dataset = FB15k237()

train_tf = dataset.training
valid_tf = dataset.validation
test_tf = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations

print(f"Training entities : {num_entities}")
print(f"Original relations: {num_relations}")

# Verify validation/test entity IDs
for split_name, split_tf in [("validation", valid_tf), ("test", test_tf)]:
    min_id = int(split_tf.mapped_triples[:, [0, 2]].min().item())
    max_id = int(split_tf.mapped_triples[:, [0, 2]].max().item())
    print(f"{split_name.capitalize()} entity IDs: {min_id} -> {max_id}")
    if min_id < 0 or max_id >= num_entities:
        raise ValueError(f"{split_name} contains entity IDs outside the training vocabulary.")

# 2. LOAD ENTITY TEXT
print("\n[2/6] Loading FB15k-237 entity descriptions...")

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(TEXT_NAME_URL, sep="\t", header=None, names=["mid", "name"], dtype=str, on_bad_lines="skip")
    mid_to_name = dict(zip(df_name["mid"], df_name["name"]))
except Exception as exc:
    print(f"WARNING: Failed to load entity names: {exc}")

try:
    df_desc = pd.read_csv(TEXT_DESC_URL, sep="\t", header=None, names=["mid", "desc"], dtype=str, on_bad_lines="skip")
    mid_to_desc = dict(zip(df_desc["mid"], df_desc["desc"]))
except Exception as exc:
    print(f"WARNING: Failed to load descriptions: {exc}")

entity_to_id = train_tf.entity_to_id
id_to_mid = {int(entity_id): str(mid) for mid, entity_id in entity_to_id.items()}

entity_id_to_text = []
for entity_id in range(num_entities):
    mid = id_to_mid.get(entity_id)
    if mid is None:
        text = f"Freebase entity {entity_id}"
    elif mid in mid_to_desc and pd.notna(mid_to_desc[mid]) and str(mid_to_desc[mid]).strip():
        text = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]) and str(mid_to_name[mid]).strip():
        text = str(mid_to_name[mid])
    else:
        text = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_text.append(text)

print(f"Prepared text for {len(entity_id_to_text):,} entities.")

# 3. LOAD APSP + INCREMENTAL PCA
print("\n[3/6] Loading APSP files and fitting Incremental PCA...")

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"APSP directory not found:\n{DATA_DIR}")

csv_files = [path for path in glob.glob(os.path.join(DATA_DIR, "*.csv")) if "shortest_paths_" in os.path.basename(path)]
if not csv_files:
    raise FileNotFoundError(f"No shortest-path CSV files found in:\n{DATA_DIR}")

def batch_start_id(path: str) -> int:
    match = re.search(r"batch_(\d+)_to_", os.path.basename(path))
    if match is None:
        raise ValueError(f"Cannot parse batch ID from {os.path.basename(path)}")
    return int(match.group(1))

csv_files.sort(key=batch_start_id)
print(f"Found {len(csv_files)} APSP batch files.")

def load_similarity_matrix(csv_path: str) -> np.ndarray:
    distances = pd.read_csv(csv_path, header=None).values.astype(np.float32)
    finite_mask = np.isfinite(distances) & (distances >= 0)
    
    if not finite_mask.any():
        raise ValueError(f"No valid distances found in {csv_path}")
        
    global_max_distance = float(distances[finite_mask].max())
    distances[~finite_mask] = global_max_distance + 1.0  # Unreachable pairs
    
    similarities = np.exp(-distances)
    row_max = similarities.max(axis=1, keepdims=True)
    similarities /= np.where(row_max > 0, row_max, 1.0)
    
    return similarities.astype(np.float32)

ipca = IncrementalPCA(n_components=PCA_COMPONENTS, batch_size=PCA_BATCH_SIZE)

for idx, csv_path in enumerate(csv_files, start=1):
    similarities = load_similarity_matrix(csv_path)
    print(f"  PCA fit {idx:>3}/{len(csv_files)} | {os.path.basename(csv_path)} | shape={similarities.shape}")
    ipca.partial_fit(similarities)
    del similarities
    gc.collect()

print("Incremental PCA fitting complete.")

compressed_blocks = []
for idx, csv_path in enumerate(csv_files, start=1):
    similarities = load_similarity_matrix(csv_path)
    compressed = ipca.transform(similarities)
    compressed_blocks.append(compressed.astype(np.float32))
    print(f"  PCA transform {idx:>3}/{len(csv_files)}")
    del similarities
    gc.collect()

apsp_struct_emb = torch.from_numpy(np.vstack(compressed_blocks)).float()
del compressed_blocks
gc.collect()

if apsp_struct_emb.shape[0] != num_entities:
    raise ValueError(f"APSP/entity vocabulary mismatch: APSP rows = {apsp_struct_emb.shape[0]}, Training nodes = {num_entities}")

print(f"Compiled structural representation: {tuple(apsp_struct_emb.shape)}")

# 4. FILTERED EVALUATION MAPS
print("\n[4/6] Building filtered evaluation maps...")

def build_filter_maps(triples_tensor: torch.Tensor):
    tail_map, head_map = {}, {}
    for h, r, t in triples_tensor.cpu().numpy():
        h, r, t = int(h), int(r), int(t)
        tail_map.setdefault((h, r), set()).add(t)
        head_map.setdefault((r, t), set()).add(h)
    return tail_map, head_map

all_triples = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_triples)

print(f"Tail filter groups: {len(filter_tails):,}")
print(f"Head filter groups: {len(filter_heads):,}")

# 5. CONTROLLED STRUCTURALBERT ABLATION MODEL
class StructuralBERTAblation(nn.Module):
    def __init__(self, use_text: bool, use_struct: bool, fusion: str, apsp_struct_emb: torch.Tensor, 
                 num_entities: int, num_relations: int, device: torch.device):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion
        self.embedding_dim = EMBEDDING_DIM

        # Relation embeddings
        self.relation_emb = nn.Embedding(num_relations, self.embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        # Structural encoder
        self.struct_proj = nn.Sequential(
            nn.Linear(apsp_struct_emb.shape[1], self.embedding_dim),
            nn.GELU(),
            nn.LayerNorm(self.embedding_dim),
            nn.Linear(self.embedding_dim, self.embedding_dim),
        )
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=False)

        # Frozen RoBERTa encoder
        if self.use_text:
            self.tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
            self.bert = AutoModel.from_pretrained(BERT_MODEL).to(device)
            for parameter in self.bert.parameters():
                parameter.requires_grad = False
            self.bert.eval()
            
            bert_hidden = self.bert.config.hidden_size
            self.text_proj = nn.Linear(bert_hidden, self.embedding_dim)
            self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden, dtype=torch.float32), persistent=False)

        # Residual entity representation
        self.entity_residual = nn.Embedding(num_entities, self.embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        # Fusion Layers
        active_modalities = int(use_text) + int(use_struct)
        if fusion == "glu":
            if active_modalities != 2:
                raise ValueError("GeGLU requires both text and structure.")
            fusion_input_dim = 2 * self.embedding_dim
            self.glu_value = nn.Linear(fusion_input_dim, self.embedding_dim)
            self.glu_gate = nn.Linear(fusion_input_dim, self.embedding_dim)
            self.glu_mix = nn.Linear(self.embedding_dim, self.embedding_dim)
        elif fusion == "add":
            if active_modalities != 2:
                raise ValueError("Add-Fusion requires both text and structure.")
        elif fusion == "none":
            if active_modalities > 1:
                raise ValueError("Fusion='none' is only valid with one or zero modalities.")
        else:
            raise ValueError(f"Unsupported fusion mode: {fusion}")

        self.to(device)

    @torch.no_grad()
    def encode_entity_texts(self, texts, batch_size: int = 128):
        if not self.use_text:
            return
        self.bert.eval()
        encoded_blocks = []
        for start in range(0, self.num_entities, batch_size):
            text_batch = texts[start : start + batch_size]
            tokens = self.tokenizer(text_batch, padding=True, truncation=True, max_length=TEXT_MAX_LENGTH, return_tensors="pt")
            tokens = {key: value.to(self.device) for key, value in tokens.items()}
            cls_embedding = self.bert(**tokens).last_hidden_state[:, 0, :]
            encoded_blocks.append(cls_embedding.detach())

        bert_embeddings = torch.cat(encoded_blocks, dim=0)
        if bert_embeddings.shape[0] != self.num_entities:
            raise RuntimeError("Text/entity count mismatch.")
            
        self.bert_cache.copy_(bert_embeddings)
        del encoded_blocks, bert_embeddings
        gc.collect()

    def entity_rep(self, ids: torch.Tensor):
        ids = ids.to(self.device, non_blocking=True)
        batch_size = ids.shape[0]

        text = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(batch_size, self.embedding_dim, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(text)

        if self.use_text and self.use_struct:
            if self.fusion == "add":
                fused = text + struct
            elif self.fusion == "glu":
                combined = torch.cat([text, struct], dim=-1)
                value = self.glu_value(combined)
                gate = self.glu_gate(combined)
                fused = self.glu_mix(value * F.gelu(gate))
            else:
                raise RuntimeError("Invalid multimodal fusion.")
        elif self.use_text:
            fused = text
        elif self.use_struct:
            fused = struct
        else:
            fused = torch.zeros(batch_size, self.embedding_dim, device=self.device)

        return fused + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb, r_emb, t_emb = self.entity_rep(h), self.relation_emb(r), self.entity_rep(t)
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        return (h_re * r_re * t_re + h_im * r_re * t_im + h_re * r_im * t_im - h_im * r_im * t_re).sum(dim=-1)

# 6. RECIPROCAL TRAINING DATA
print("\n[5/6] Preparing reciprocal training triples...")

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]
inverse_train[:, 2] = mapped_train[:, 0]
inverse_train[:, 1] = mapped_train[:, 1] + num_relations

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True, pin_memory=torch.cuda.is_available())

# 7. BUILD ALL ENTITY REPRESENTATIONS FOR EVALUATION
@torch.no_grad()
def build_all_entity_embeddings(model):
    model.eval()
    all_embeddings = torch.empty(model.num_entities, model.embedding_dim, device=model.device)
    for start in range(0, model.num_entities, ENTITY_EMB_BATCH_SIZE):
        ids = torch.arange(start, min(start + ENTITY_EMB_BATCH_SIZE, model.num_entities), device=model.device)
        all_embeddings[start : start + len(ids)] = model.entity_rep(ids)
    return all_embeddings

# 8. STANDARD FILTERED HEAD + TAIL EVALUATION
@torch.no_grad()
def evaluate_model(model, eval_tf, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    all_entity_emb = build_all_entity_embeddings(model)
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)

    tail_ranks, head_ranks = [], []

    for start in range(0, triples.shape[0], batch_size):
        batch = triples[start : start + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, t_emb, r_emb = all_entity_emb[h], all_entity_emb[t], model.relation_emb(r)

        # Tail prediction
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))

        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            filtered = [entity for entity in filter_tails.get((hi, ri), set()) if entity != ti]
            if filtered:
                scores_tail[i, filtered] = -1e9
            
            target_score = scores_tail[i, ti].item()
            rank = (scores_tail[i] > target_score).sum().item() + 1
            tail_ranks.append(rank)

        # Head prediction
        inverse_r = r + num_relations
        inverse_r_emb = model.relation_emb(inverse_r)
        r_inv_re, r_inv_im = torch.chunk(inverse_r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))

        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            filtered = [entity for entity in filter_heads.get((ri, ti), set()) if entity != hi]
            if filtered:
                scores_head[i, filtered] = -1e9
            
            target_score = scores_head[i, hi].item()
            rank = (scores_head[i] > target_score).sum().item() + 1
            head_ranks.append(rank)

    tail_ranks = np.asarray(tail_ranks, dtype=np.int64)
    head_ranks = np.asarray(head_ranks, dtype=np.int64)
    all_ranks = np.concatenate([tail_ranks, head_ranks])

    def metrics(ranks):
        return {
            "MRR": float(np.mean(1.0 / ranks)),
            "Hits@1": float(np.mean(ranks <= 1)),
            "Hits@10": float(np.mean(ranks <= 10)),
        }

    return {"tail": metrics(tail_ranks), "head": metrics(head_ranks), "overall": metrics(all_ranks)}

# 9. CONTROLLED ABLATION RUNNER
def run_ablation(name, use_text, use_struct, fusion):
    print("\n" + "=" * 80)
    print(f"STARTING ABLATION: {name}")
    print("=" * 80)
    print(f"text={use_text} | structure={use_struct} | fusion={fusion}")

    model = StructuralBERTAblation(
        use_text=use_text, use_struct=use_struct, fusion=fusion,
        apsp_struct_emb=apsp_struct_emb, num_entities=num_entities,
        num_relations=num_relations * 2, device=DEVICE,
    )

    if use_text:
        print("Caching frozen RoBERTa representations...")
        model.encode_entity_texts(entity_id_to_text, batch_size=128)

    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_{name}.pt")
    
    best_val_mrr = -1.0
    patience_counter = 0
    start_time = time.time()

    # Training Loop
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        num_batches = 0

        for (batch_triples,) in train_loader:
            batch_triples = batch_triples.to(DEVICE, non_blocking=True)
            h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

            neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEGATIVES), device=DEVICE)
            pos_scores = model.score_triples(h, r, t_pos)

            h_rep = h.unsqueeze(1).expand(-1, NUM_NEGATIVES).reshape(-1)
            r_rep = r.unsqueeze(1).expand(-1, NUM_NEGATIVES).reshape(-1)
            neg_scores = model.score_triples(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEGATIVES)

            pos_loss = -F.logsigmoid(pos_scores).mean()
            neg_weights = torch.softmax(neg_scores, dim=-1).detach()
            neg_loss = -(neg_weights * F.logsigmoid(-neg_scores)).sum(dim=-1).mean()

            # CORRECT BATCH-WISE L2 REGULARIZATION
            h_res = model.entity_residual(h)
            r_emb = model.relation_emb(r)
            t_pos_res = model.entity_residual(t_pos)
            t_neg_res = model.entity_residual(neg_t.reshape(-1))

            batch_len = h.size(0)
            l2_reg = (REG_WEIGHT / batch_len) * (
                h_res.norm(p=2) ** 2 + 
                r_emb.norm(p=2) ** 2 + 
                t_pos_res.norm(p=2) ** 2 + 
                t_neg_res.norm(p=2) ** 2
            )

            loss = pos_loss + neg_loss + l2_reg

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            total_loss += float(loss.item())
            num_batches += 1

        avg_loss = total_loss / max(num_batches, 1)

        # Validation
        validation = evaluate_model(model, valid_tf)
        val_mrr = validation["overall"]["MRR"]
        scheduler.step(val_mrr)

        print(f"Epoch {epoch:03d} | Loss {avg_loss:.4f} | Val MRR {val_mrr:.4f} | Val Hits@10 {validation['overall']['Hits@10']:.4f}")

        # Best checkpoint
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> Best checkpoint saved (MRR={best_val_mrr:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    training_minutes = (time.time() - start_time) / 60.0

    # Restore best checkpoint and test
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    test_results = evaluate_model(model, test_tf)
    overall = test_results["overall"]

    print(f"\n{name} TEST RESULTS")
    print(f"  Tail MRR    : {test_results['tail']['MRR']:.4f}")
    print(f"  Head MRR    : {test_results['head']['MRR']:.4f}")
    print(f"  Overall MRR : {overall['MRR']:.4f}")
    print(f"  Hits@1      : {overall['Hits@1']:.4f}")
    print(f"  Hits@10     : {overall['Hits@10']:.4f}")
    print(f"  Train time  : {training_minutes:.2f} min")

    result = {
        "MRR": overall["MRR"],
        "Hits@1": overall["Hits@1"],
        "Hits@10": overall["Hits@10"],
        "Tail_MRR": test_results["tail"]["MRR"],
        "Head_MRR": test_results["head"]["MRR"],
        "Best_Val_MRR": best_val_mrr,
        "Training_Minutes": training_minutes,
    }

    # Free model/GPU memory before next run
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return result

# 10. CONTROLLED EXPERIMENTS
results = {}

results["Residual-Only"] = run_ablation("Residual-Only", False, False, "none")
results["Residual+Text"] = run_ablation("Residual-Text", True, False, "none")
results["Residual+Structure"] = run_ablation("Residual-Structure", False, True, "none")
results["Add-Fusion"] = run_ablation("Add-Fusion", True, True, "add")
results["GeGLU"] = run_ablation("GeGLU", True, True, "glu")

# 11. FINAL RESULTS
print("\n" + "=" * 80)
print("FINAL CONTROLLED ABLATION RESULTS — FB15k-237")
print("=" * 80)
print(f"{'Configuration':<24}{'MRR':>10}{'Hits@1':>10}{'Hits@10':>10}")
print("-" * 80)
for name, metrics in results.items():
    print(f"{name:<24}{metrics['MRR']:>10.4f}{metrics['Hits@1']:>10.4f}{metrics['Hits@10']:>10.4f}")
print("-" * 80)

# 12. DIRECT EFFECT CALCULATIONS
if "Residual-Only" in results and "Residual+Structure" in results:
    structural_gain = results["Residual+Structure"]["MRR"] - results["Residual-Only"]["MRR"]
    print(f"\nAPSP structural gain over Residual-Only: {structural_gain:+.4f} MRR")

if "Residual+Text" in results and "Add-Fusion" in results:
    structure_plus_text_gain = results["Add-Fusion"]["MRR"] - results["Residual+Text"]["MRR"]
    print(f"Structural gain added to Residual+Text: {structure_plus_text_gain:+.4f} MRR")

if "Add-Fusion" in results and "GeGLU" in results:
    geglu_gain = results["GeGLU"]["MRR"] - results["Add-Fusion"]["MRR"]
    print(f"GeGLU gain over Add-Fusion: {geglu_gain:+.4f} MRR")

# 13. SAVE RESULTS
results_df = pd.DataFrame(results).T
results_df.to_csv(RESULTS_FILE)

print(f"\nResults saved to:\n{RESULTS_FILE}")
print("\nFB15k-237 ablation study complete.")

Writing fb15k237_ablation_study.py


In [2]:
!pip install pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 968.3 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.2/496.2 kB 21.2 MB/s eta 0:00:00


In [3]:
!time python fb15k237_ablation_study.py

StructuralBERT — FB15k-237 Controlled Ablation Study
Device : cuda
Seed   : 42

[1/6] Loading FB15k-237...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Training entities : 14505
Original relations: 237
Validation entity IDs: 0 -> 14504
Test entity IDs: 2 -> 14504

[2/6] Loading FB15k-237 entity descriptions...
Prepared text for 14,505 entities.

[3/6] Loading APSP files and fitting Incremental PCA...
Found 6 APSP batch files.
  PCA fit   1/6 | shortest_paths_batch_0_to_2499.csv | shape=(2500, 14505)
  PCA fit   2/6 | shortest_paths_batch_2500_to_4999.csv | shape=(2500, 14505)
  PCA fit   3/6 | shortest_paths_batch_5000_to_7499.csv | shape=(2500, 14505

In [16]:
%%writefile fb15k237_stv_final.py
import time
import os
import glob
import re
import gc
import urllib.request
import tarfile
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from pykeen.datasets import PathDataset
from sklearn.decomposition import IncrementalPCA

# 1. LOAD FB15k-237 DATASET & TEXT DESCRIPTIONS
print("\n[1/6] Downloading FB15k-237 manually to bypass PyKEEN 404 error...")

if not os.path.exists("FB15k-237.zip") and not os.path.exists("Release"):
    print("Downloading dataset archive...")
    urllib.request.urlretrieve("https://download.microsoft.com/download/8/7/0/8700516A-AB3D-4850-B4BB-805C515AECE1/FB15K-237.2.zip", "FB15k-237.zip")

if not os.path.exists("Release"):
    print("Extracting dataset archive...")
    with zipfile.ZipFile("FB15k-237.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

# Locate the extracted text files dynamically
train_path = glob.glob("Release/**/train.txt", recursive=True)[0]
valid_path = glob.glob("Release/**/valid.txt", recursive=True)[0]
test_path = glob.glob("Release/**/test.txt", recursive=True)[0]

dataset = PathDataset(
    training_path=train_path,
    validation_path=valid_path,
    testing_path=test_path,
)

train_tf = dataset.training
valid_tf = dataset.validation
test_tf  = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations
print(f"Entities: {num_entities} | Relations: {num_relations}")

print("\n[2/6] Fetching and mapping textual descriptions for FB15k-237...")
url_name = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2text.txt"
url_desc = "https://raw.githubusercontent.com/yao8839836/kg-bert/master/data/FB15k-237/entity2textlong.txt"

mid_to_name, mid_to_desc = {}, {}
try:
    df_name = pd.read_csv(url_name, sep="\t", header=None, names=["mid", "name"], dtype=str, on_bad_lines="skip")
    mid_to_name = dict(zip(df_name["mid"], df_name["name"]))
except Exception as exc: print(f"WARNING: Failed to load entity names: {exc}")

try:
    df_desc = pd.read_csv(url_desc, sep="\t", header=None, names=["mid", "desc"], dtype=str, on_bad_lines="skip")
    mid_to_desc = dict(zip(df_desc["mid"], df_desc["desc"]))
except Exception as exc: print(f"WARNING: Failed to load descriptions: {exc}")

entity_mapping = getattr(dataset, "entity_to_id", None) or train_tf.entity_to_id
id_to_mid = {v: str(k) for k, v in entity_mapping.items()}

entity_id_to_label = []
for i in range(num_entities):
    mid = id_to_mid.get(i)
    if mid is None:
        text = f"Freebase entity {i}"
    elif mid in mid_to_desc and pd.notna(mid_to_desc[mid]) and str(mid_to_desc[mid]).strip():
        text = str(mid_to_desc[mid])
    elif mid in mid_to_name and pd.notna(mid_to_name[mid]) and str(mid_to_name[mid]).strip():
        text = str(mid_to_name[mid])
    else:
        text = str(mid).replace("_", " ").replace("/", " ").strip()
    entity_id_to_label.append(text)

print(f" -> Text mapping complete. Example: {entity_id_to_label[0][:80]}...")

# 2. LOAD & PROCESS APSP CSVs (FB15k-237)
print("\n[3/6] Processing APSP CSVs with High-Capacity IncrementalPCA...")
DATA_DIR = "/kaggle/input/datasets/arafahmed99/apsp-fb15k-237-data"
all_csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

if not all_csv_files:
    raise FileNotFoundError(f"Could not find any CSV files in {DATA_DIR}")

all_csv_files.sort(key=lambda x: int(re.search(r'batch_(\d+)_to_', os.path.basename(x)).group(1)))

print(f" -> Found {len(all_csv_files)} batch files. Fitting Incremental PCA...")
ipca = IncrementalPCA(n_components=512, batch_size=10000)

for i, f in enumerate(all_csv_files):
    df = pd.read_csv(f, header=None)
    D_batch = df.values.astype(np.float32)
    D_batch[~np.isfinite(D_batch)] = 100.0
    
    S_batch = np.exp(-1.0 * D_batch)
    row_max = S_batch.max(axis=1, keepdims=True)
    S_batch = S_batch / np.where(row_max == 0, 1e-9, row_max)
    
    ipca.partial_fit(S_batch)
    del df, D_batch, S_batch
    gc.collect()

print(" -> Incremental PCA Fitted! Applying transform...")

S_compressed_list = []
for f in all_csv_files:
    df = pd.read_csv(f, header=None)
    D_batch = df.values.astype(np.float32)
    D_batch[~np.isfinite(D_batch)] = 100.0
    
    S_batch = np.exp(-1.0 * D_batch)
    row_max = S_batch.max(axis=1, keepdims=True)
    S_batch = S_batch / np.where(row_max == 0, 1e-9, row_max)
    
    compressed = ipca.transform(S_batch)
    S_compressed_list.append(compressed)
    
    del df, D_batch, S_batch
    gc.collect()

apsp_struct_emb = torch.tensor(np.vstack(S_compressed_list), dtype=torch.float32)
del S_compressed_list
gc.collect()

print(f" -> PCA Complete. apsp_struct_emb shape: {apsp_struct_emb.shape}")

# 3. MODEL ARCHITECTURE
class StructuralBERT_Final(nn.Module):
    def __init__(self, num_entities, num_relations, apsp_struct_emb, bert_model_name="roberta-base", embedding_dim=512, fusion="glu", use_text=True, use_struct=True, device=torch.device("cpu")):
        super().__init__()
        self.device = device
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.use_text = use_text
        self.use_struct = use_struct
        self.fusion = fusion

        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        struct_dim = apsp_struct_emb.shape[1]
        
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", apsp_struct_emb.clone(), persistent=True)

        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert = AutoModel.from_pretrained(bert_model_name).to(self.device)
        for p in self.bert.parameters(): p.requires_grad = False

        bert_hidden = self.bert.config.dim if hasattr(self.bert.config, "dim") else self.bert.config.hidden_size
        self.text_proj = nn.Linear(bert_hidden, embedding_dim)
        
        self.register_buffer("bert_cache", torch.zeros(num_entities, bert_hidden), persistent=True)
        self._bert_ready = False

        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        nn.init.xavier_uniform_(self.entity_residual.weight)

        if fusion in ("concat", "gate", "glu"):
            in_dim = (embedding_dim if use_text else 0) + (embedding_dim if use_struct else 0)
            self.fusion_layer = nn.Linear(in_dim, embedding_dim)
            self.glu_value = nn.Linear(in_dim, embedding_dim)
            self.glu_gate = nn.Linear(in_dim, embedding_dim)
            self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

        self.to(self.device)

    @torch.no_grad()
    def encode_entity_texts(self, labels, batch_size=128, max_length=64):
        self.bert.eval()
        embs = []
        for i in range(0, self.num_entities, batch_size):
            chunk = labels[i:i + batch_size]
            tok = self.tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(self.device)
            out = self.bert(**tok).last_hidden_state[:, 0, :]
            embs.append(out) 
        self.bert_cache.copy_(torch.cat(embs, dim=0).detach())
        self._bert_ready = True

    def _fuse(self, text_emb, struct_emb):
        parts = []
        if self.use_text: parts.append(text_emb)
        if self.use_struct: parts.append(struct_emb)

        if len(parts) == 1: return parts[0]
        if self.fusion == "add": return sum(parts)

        combined = torch.cat(parts, dim=-1)
        
        if self.fusion == "glu":
            value = self.glu_value(combined)
            gate = self.glu_gate(combined)
            fused = value * torch.nn.functional.gelu(gate)
            return self.glu_mix(fused)
        
        fused = self.fusion_layer(combined)
        if self.fusion == "concat": return torch.tanh(fused)
        elif self.fusion == "gate": return torch.sigmoid(fused) * parts[0] + (1.0 - torch.sigmoid(fused)) * parts[1]

    def entity_rep(self, ids=None):
        if ids is None: ids = torch.arange(self.num_entities, device=self.device)
        else: ids = ids.to(self.device)

        txt = self.text_proj(self.bert_cache[ids]) if self.use_text else torch.zeros(len(ids), self.text_proj.out_features, device=self.device)
        struct = self.struct_proj(self.struct_cache[ids]) if self.use_struct else torch.zeros_like(txt)
        return self._fuse(txt, struct) + self.entity_residual(ids)

    def score_triples(self, h, r, t):
        h_emb = self.entity_rep(h)
        r_emb = self.relation_emb(r)
        t_emb = self.entity_rep(t)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)

        scores = (h_re * r_re * t_re +
                  h_im * r_re * t_im +
                  h_re * r_im * t_im -
                  h_im * r_im * t_re).sum(dim=-1)
        return scores

    def forward(self, h, r, t):
        return self.score_triples(h, r, t)

# 4. FAST VECTORIZED HEAD+TAIL EVALUATOR 
def build_filter_maps(all_triples_tensor):
    to_tails, to_heads = {}, {}
    for h, r, t in all_triples_tensor.cpu().numpy():
        to_tails.setdefault((int(h), int(r)), set()).add(int(t))
        to_heads.setdefault((int(r), int(t)), set()).add(int(h))
    return to_tails, to_heads

all_mapped = torch.cat([train_tf.mapped_triples, valid_tf.mapped_triples, test_tf.mapped_triples], dim=0)
filter_tails, filter_heads = build_filter_maps(all_mapped)

@torch.no_grad()
def evaluate_model_head_tail(model, eval_tf, filter_tails, filter_heads, num_orig_rels, batch_size=256):
    model.eval()
    triples = eval_tf.mapped_triples.to(model.device)
    num_samples = triples.shape[0]

    all_entity_emb = torch.zeros(model.num_entities, model.entity_residual.embedding_dim, device=model.device)
    for i in range(0, model.num_entities, 2048):
        chunk_ids = torch.arange(i, min(i+2048, model.num_entities), device=model.device)
        all_entity_emb[chunk_ids] = model.entity_rep(chunk_ids)
        
    all_e_re, all_e_im = torch.chunk(all_entity_emb, 2, dim=-1)
    
    tail_ranks, head_ranks = [], []

    for start_idx in range(0, num_samples, batch_size):
        batch = triples[start_idx:start_idx + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        h_emb, r_emb, t_emb = all_entity_emb[h], model.relation_emb(r), all_entity_emb[t]
        
        r_inv = r + num_orig_rels
        r_inv_emb = model.relation_emb(r_inv)

        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        r_inv_re, r_inv_im = torch.chunk(r_inv_emb, 2, dim=-1)

        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        scores_tail = torch.matmul(hr_re, all_e_re.transpose(0, 1)) + torch.matmul(hr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_tails.get((hi, ri), set()) if e != ti]
            if mask_idx: scores_tail[i, mask_idx] = -1e9
            tail_ranks.append((scores_tail[i] > scores_tail[i, ti].item()).sum().item() + 1)

        tr_re = t_re * r_inv_re - t_im * r_inv_im
        tr_im = t_re * r_inv_im + t_im * r_inv_re
        scores_head = torch.matmul(tr_re, all_e_re.transpose(0, 1)) + torch.matmul(tr_im, all_e_im.transpose(0, 1))
        
        for i in range(len(batch)):
            hi, ri, ti = int(h[i].item()), int(r[i].item()), int(t[i].item())
            mask_idx = [e for e in filter_heads.get((ri, ti), set()) if e != hi]
            if mask_idx: scores_head[i, mask_idx] = -1e9
            head_ranks.append((scores_head[i] > scores_head[i, hi].item()).sum().item() + 1)

    tail_ranks, head_ranks = np.array(tail_ranks), np.array(head_ranks)
    def get_metrics(ranks): return {"MRR": np.mean(1.0/ranks), "Hits@1": np.mean(ranks<=1), "Hits@10": np.mean(ranks<=10)}
    return get_metrics(tail_ranks), get_metrics(head_ranks), get_metrics(np.concatenate([tail_ranks, head_ranks]))

# 5. INITIALIZATION & TRAINING LOOP
print("\n[4/6] Initializing FB15k-237 Model with RoBERTa...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = StructuralBERT_Final(
    num_entities=num_entities, 
    num_relations=num_relations * 2,
    apsp_struct_emb=apsp_struct_emb, 
    embedding_dim=512,  
    fusion="glu",  
    use_text=True, 
    use_struct=True, 
    device=device
)

print("Encoding entity labels with Transformer (Offline Cache)...")
model.encode_entity_texts(entity_id_to_label, batch_size=256)

BATCH_SIZE, NUM_NEG, LR, NUM_EPOCHS, PATIENCE = 2048, 64, 3e-4, 100, 10
ALPHA = 1.0  
REG_WEIGHT = 1e-4  

mapped_train = train_tf.mapped_triples
inverse_train = mapped_train.clone()
inverse_train[:, 0] = mapped_train[:, 2]               
inverse_train[:, 2] = mapped_train[:, 0]               
inverse_train[:, 1] = mapped_train[:, 1] + num_relations 

extended_train_triples = torch.cat([mapped_train, inverse_train], dim=0)
train_loader = DataLoader(TensorDataset(extended_train_triples), batch_size=BATCH_SIZE, shuffle=True)

optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("\n[5/6] === START TRAINING: FB15k-237 SOTA ===\n")
best_mrr, patience_counter = -1.0, 0
t0 = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss, nbatches = 0.0, 0

    for (batch_triples,) in train_loader:
        batch_triples = batch_triples.to(device)
        h, r, t_pos = batch_triples[:, 0], batch_triples[:, 1], batch_triples[:, 2]

        neg_t = torch.randint(0, num_entities, size=(len(t_pos), NUM_NEG), device=device)
        
        pos_scores = model(h, r, t_pos)
        h_rep = h.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        r_rep = r.unsqueeze(1).expand(-1, NUM_NEG).reshape(-1)
        neg_scores = model(h_rep, r_rep, neg_t.reshape(-1)).reshape(-1, NUM_NEG)

        pos_loss = -torch.nn.functional.logsigmoid(pos_scores).mean()
        neg_weights = torch.softmax(neg_scores * ALPHA, dim=-1).detach()
        neg_loss = -(neg_weights * torch.nn.functional.logsigmoid(-neg_scores)).sum(dim=-1).mean()
        
        h_res = model.entity_residual(h)
        r_emb = model.relation_emb(r)
        t_pos_res = model.entity_residual(t_pos)
        t_neg_res = model.entity_residual(neg_t.reshape(-1))

        batch_len = h.size(0)
        l2_reg = (REG_WEIGHT / batch_len) * (
            h_res.norm(p=2)**2 + 
            r_emb.norm(p=2)**2 + 
            t_pos_res.norm(p=2)**2 + 
            t_neg_res.norm(p=2)**2
        )

        loss = pos_loss + neg_loss + l2_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        nbatches += 1

    avg_train_loss = total_loss / max(nbatches, 1)

    _, _, val_metrics = evaluate_model_head_tail(model, valid_tf, filter_tails, filter_heads, num_relations)
    scheduler.step(val_metrics["MRR"])

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Valid MRR: {val_metrics['MRR']:.4f} | Hits@10: {val_metrics['Hits@10']:.4f}")

    if val_metrics["MRR"] > best_mrr:
        best_mrr = val_metrics["MRR"]
        patience_counter = 0
        torch.save(model.state_dict(), "best_fb15k237_final.pt")
        print("  --> New best model checkpoint saved!")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

print(f"\nTraining completed in {(time.time() - t0)/60:.2f} minutes.")

# 6. FINAL TEST EVALUATION
print("\n[6/6] === FINAL TEST RESULTS (Standard Filtered) ===")
if os.path.exists("best_fb15k237_final.pt"):
    model.load_state_dict(torch.load("best_fb15k237_final.pt", map_location=device))

tail_res, head_res, overall_res = evaluate_model_head_tail(model, test_tf, filter_tails, filter_heads, num_relations)

print("="*50)
print(f"Metric        | Tail-Only | Head-Only | Standard (Avg)")
print(f"MRR           |  {tail_res['MRR']:.4f}   |  {head_res['MRR']:.4f}   |  {overall_res['MRR']:.4f}")
print(f"Hits@1        |  {tail_res['Hits@1']:.4f}   |  {head_res['Hits@1']:.4f}   |  {overall_res['Hits@1']:.4f}")
print(f"Hits@10       |  {tail_res['Hits@10']:.4f}   |  {head_res['Hits@10']:.4f}   |  {overall_res['Hits@10']:.4f}")
print("="*50)

Writing fb15k237_stv_final.py


In [17]:
!time python fb15k237_stv_final.py


[1/6] Downloading FB15k-237 manually to bypass PyKEEN 404 error...
Extracting dataset archive...
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out
Entities: 14505 | Relations: 237

[2/6] Fetching and mapping textual descriptions for FB15k-237...
 -> Text mapping complete. Example: Denton is a city in the U.S. state of Texas and the county seat of Denton County...

[3/6] Processing APSP CSVs with High-Capacity IncrementalPCA...
 -> Found 6 batch files. Fitting Incremental PCA...
 -> Incremental PCA Fitted! Applying transform...
 -> PCA Complete. apsp_struct_emb shape: torch.Size([14505, 512])

[4/6] Initializing FB15k-237 Model with RoBERTa...
Using device

now going for latecy testing with other models

In [18]:
%%writefile hardware_latency_fb15k237.py

import torch
import torch.nn as nn
import time
import numpy as np
import os


NUM_ENTITIES = 14505
NUM_RELATIONS = 474 # 237 original * 2 (inverse)
EMBEDDING_DIM = 512
CHECKPOINT_PATH = "best_fb15k237_final.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Hardware Latency Benchmark on: {DEVICE}")

# 1. THE DEPLOYED MODEL (Fast Query Evaluation)
class DeployedStructuralBERT(nn.Module):
    """
    This represents StructuralBERT in its final deployed state.
    The GeGLU, RoBERTa, and PCA components have been completely compiled 
    away into a single, static entity lookup table.
    """
    def __init__(self, final_entity_embeddings, relation_embeddings):
        super().__init__()
        # Load the precompiled, fused embeddings directly into standard lookups
        self.e_entity = nn.Embedding.from_pretrained(final_entity_embeddings, freeze=True)
        self.relation_emb = nn.Embedding.from_pretrained(relation_embeddings, freeze=True)
        
    def forward(self, h_idx, r_idx):
        h_emb = self.e_entity(h_idx)      # Shape: [Batch, 512]
        r_emb = self.relation_emb(r_idx)  # Shape: [Batch, 512]
        t_emb = self.e_entity.weight      # Shape: [14505, 512]
        
        # ComplEx Scoring Math
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        
        # Fast 1-to-N matrix multiplication against all 14,505 entities simultaneously
        scores = torch.matmul(hr_re, t_re.t()) + torch.matmul(hr_im, t_im.t())
        return scores

# 2. MODEL DEFINITION (To load the weights)
class StructuralBERT_Final(nn.Module):
    def __init__(self):
        super().__init__()
        self.use_text = True
        self.use_struct = True
        self.fusion = "glu"
        self.relation_emb = nn.Embedding(NUM_RELATIONS, EMBEDDING_DIM)
        
        self.struct_proj = nn.Sequential(
            nn.Linear(512, EMBEDDING_DIM), nn.GELU(),
            nn.LayerNorm(EMBEDDING_DIM), nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM)
        )
        self.register_buffer("struct_cache", torch.zeros(NUM_ENTITIES, 512))
        
        self.text_proj = nn.Linear(768, EMBEDDING_DIM)
        self.register_buffer("bert_cache", torch.zeros(NUM_ENTITIES, 768))
        
        self.entity_residual = nn.Embedding(NUM_ENTITIES, EMBEDDING_DIM)
        
        self.glu_value = nn.Linear(1024, EMBEDDING_DIM)
        self.glu_gate = nn.Linear(1024, EMBEDDING_DIM)
        self.glu_mix = nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM)

    def _fuse(self, txt, struct):
        combined = torch.cat([txt, struct], dim=-1)
        value = self.glu_value(combined)
        gate = self.glu_gate(combined)
        fused = value * torch.nn.functional.gelu(gate)
        return self.glu_mix(fused)

    def entity_rep(self):
        ids = torch.arange(NUM_ENTITIES, device=self.struct_cache.device)
        txt = self.text_proj(self.bert_cache[ids])
        struct = self.struct_proj(self.struct_cache[ids])
        return self._fuse(txt, struct) + self.entity_residual(ids)

# 3. LOAD & COMPILE
print(f"\n[1/3] Loading trained weights from {CHECKPOINT_PATH}...")
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

# Load full model
full_model = StructuralBERT_Final().to(DEVICE)
full_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True), strict=False)
full_model.eval()

print(" -> Compiling modalities into static entity representation...")
with torch.no_grad():
    final_entities = full_model.entity_rep()
    final_relations = full_model.relation_emb.weight.clone()

deployed_model = DeployedStructuralBERT(final_entities, final_relations).to(DEVICE)
deployed_model.eval()

# Free up VRAM from the heavy model
del full_model
torch.cuda.empty_cache()

# 4. BENCHMARK 1: SINGLE-QUERY LATENCY (Real-time serving)
print("\n[2/3] --- BENCHMARK 1: Single Query Latency (Batch Size = 1) ---")
h_single = torch.tensor([42]).to(DEVICE)
r_single = torch.tensor([5]).to(DEVICE)

# Warmup GPU
for _ in range(100):
    _ = deployed_model(h_single, r_single)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
latencies = []

with torch.no_grad():
    for _ in range(1000):
        start_event.record()
        scores = deployed_model(h_single, r_single)
        end_event.record()
        torch.cuda.synchronize()
        latencies.append(start_event.elapsed_time(end_event))

print(f"Average Latency per query : {np.mean(latencies):.3f} ms")
print(f"99th Percentile (p99)     : {np.percentile(latencies, 99):.3f} ms")

# 5. BENCHMARK 2: THROUGHPUT (Batch Processing)
print("\n[3/3] --- BENCHMARK 2: Throughput (Batch Size = 1024) ---")
BATCH_SIZE = 1024
h_batch = torch.randint(0, NUM_ENTITIES, (BATCH_SIZE,)).to(DEVICE)
r_batch = torch.randint(0, NUM_RELATIONS, (BATCH_SIZE,)).to(DEVICE)

for _ in range(50):
    _ = deployed_model(h_batch, r_batch)
torch.cuda.synchronize()

start_time = time.time()
num_batches = 100
with torch.no_grad():
    for _ in range(num_batches):
        _ = deployed_model(h_batch, r_batch)
torch.cuda.synchronize()
end_time = time.time()

total_time_sec = end_time - start_time
total_queries = num_batches * BATCH_SIZE

print(f"Total Queries Evaluated : {total_queries:,}")
print(f"Total Time              : {total_time_sec:.3f} seconds")
print(f"Throughput              : {total_queries / total_time_sec:,.0f} queries / second")
print(f"Peak VRAM Usage         : {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

Writing hardware_latency_fb15k237.py


In [19]:
!time python hardware_latency_fb15k237.py

Running Hardware Latency Benchmark on: cuda

[1/3] Loading trained weights from best_fb15k237_final.pt...
 -> Compiling modalities into static entity representation...

[2/3] --- BENCHMARK 1: Single Query Latency (Batch Size = 1) ---
Average Latency per query : 0.346 ms
99th Percentile (p99)     : 0.553 ms

[3/3] --- BENCHMARK 2: Throughput (Batch Size = 1024) ---
Total Queries Evaluated : 102,400
Total Time              : 0.459 seconds
Throughput              : 222,978 queries / second
Peak VRAM Usage         : 696.59 MB

real	0m5.183s
user	0m4.076s
sys	0m1.513s


In [20]:
%%writefile hardware_latency_wn18rr.py

import torch
import torch.nn as nn
import time
import numpy as np
import os

NUM_ENTITIES = 40559
NUM_RELATIONS = 22 # 11 original * 2 (inverse)
EMBEDDING_DIM = 512
CHECKPOINT_PATH = "/kaggle/working/best_wn18rr_final.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Hardware Latency Benchmark on: {DEVICE}")

# 1. THE DEPLOYED MODEL (Fast Query Evaluation)
class DeployedStructuralBERT(nn.Module):
    """
    This represents StructuralBERT in its final deployed state.
    The GeGLU, RoBERTa, and PCA components have been completely compiled 
    away into a single, static entity lookup table.
    """
    def __init__(self, final_entity_embeddings, relation_embeddings):
        super().__init__()
        # Load the precompiled, fused embeddings directly into standard lookups
        self.e_entity = nn.Embedding.from_pretrained(final_entity_embeddings, freeze=True)
        self.relation_emb = nn.Embedding.from_pretrained(relation_embeddings, freeze=True)
        
    def forward(self, h_idx, r_idx):
        h_emb = self.e_entity(h_idx)      # Shape: [Batch, 512]
        r_emb = self.relation_emb(r_idx)  # Shape: [Batch, 512]
        t_emb = self.e_entity.weight      # Shape: [40559, 512]
        
        # ComplEx Scoring Math
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        
        # Fast 1-to-N matrix multiplication against all 40,559 entities simultaneously
        scores = torch.matmul(hr_re, t_re.t()) + torch.matmul(hr_im, t_im.t())
        return scores

# 2. MODEL DEFINITION (To load the weights)
class StructuralBERT_Final(nn.Module):
    def __init__(self):
        super().__init__()
        self.use_text = True
        self.use_struct = True
        self.fusion = "glu"
        self.relation_emb = nn.Embedding(NUM_RELATIONS, EMBEDDING_DIM)
        
        self.struct_proj = nn.Sequential(
            nn.Linear(512, EMBEDDING_DIM), nn.GELU(),
            nn.LayerNorm(EMBEDDING_DIM), nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM)
        )
        self.register_buffer("struct_cache", torch.zeros(NUM_ENTITIES, 512))
        
        self.text_proj = nn.Linear(768, EMBEDDING_DIM)
        self.register_buffer("bert_cache", torch.zeros(NUM_ENTITIES, 768))
        
        self.entity_residual = nn.Embedding(NUM_ENTITIES, EMBEDDING_DIM)
        
        self.glu_value = nn.Linear(1024, EMBEDDING_DIM)
        self.glu_gate = nn.Linear(1024, EMBEDDING_DIM)
        self.glu_mix = nn.Linear(EMBEDDING_DIM, EMBEDDING_DIM)

    def _fuse(self, txt, struct):
        combined = torch.cat([txt, struct], dim=-1)
        value = self.glu_value(combined)
        gate = self.glu_gate(combined)
        fused = value * torch.nn.functional.gelu(gate)
        return self.glu_mix(fused)

    def entity_rep(self):
        ids = torch.arange(NUM_ENTITIES, device=self.struct_cache.device)
        txt = self.text_proj(self.bert_cache[ids])
        struct = self.struct_proj(self.struct_cache[ids])
        return self._fuse(txt, struct) + self.entity_residual(ids)

# 3. LOAD & COMPILE
print(f"\n[1/3] Loading trained weights from {CHECKPOINT_PATH}...")
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

# Load full model
full_model = StructuralBERT_Final().to(DEVICE)
full_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True), strict=False)
full_model.eval()

print(" -> Compiling modalities into static entity representation...")
with torch.no_grad():
    final_entities = full_model.entity_rep()
    final_relations = full_model.relation_emb.weight.clone()

deployed_model = DeployedStructuralBERT(final_entities, final_relations).to(DEVICE)
deployed_model.eval()

# Free up VRAM from the heavy model
del full_model
torch.cuda.empty_cache()

# 4. BENCHMARK 1: SINGLE-QUERY LATENCY (Real-time serving)
print("\n[2/3] --- BENCHMARK 1: Single Query Latency (Batch Size = 1) ---")
h_single = torch.tensor([42]).to(DEVICE)
r_single = torch.tensor([5]).to(DEVICE)

# Warmup GPU
for _ in range(100):
    _ = deployed_model(h_single, r_single)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
latencies = []

with torch.no_grad():
    for _ in range(1000):
        start_event.record()
        scores = deployed_model(h_single, r_single)
        end_event.record()
        torch.cuda.synchronize()
        latencies.append(start_event.elapsed_time(end_event))

print(f"Average Latency per query : {np.mean(latencies):.3f} ms")
print(f"99th Percentile (p99)     : {np.percentile(latencies, 99):.3f} ms")

# 5. BENCHMARK 2: THROUGHPUT (Batch Processing)
print("\n[3/3] --- BENCHMARK 2: Throughput (Batch Size = 1024) ---")
BATCH_SIZE = 1024
h_batch = torch.randint(0, NUM_ENTITIES, (BATCH_SIZE,)).to(DEVICE)
r_batch = torch.randint(0, NUM_RELATIONS, (BATCH_SIZE,)).to(DEVICE)

for _ in range(50):
    _ = deployed_model(h_batch, r_batch)
torch.cuda.synchronize()

start_time = time.time()
num_batches = 100
with torch.no_grad():
    for _ in range(num_batches):
        _ = deployed_model(h_batch, r_batch)
torch.cuda.synchronize()
end_time = time.time()

total_time_sec = end_time - start_time
total_queries = num_batches * BATCH_SIZE

print(f"Total Queries Evaluated : {total_queries:,}")
print(f"Total Time              : {total_time_sec:.3f} seconds")
print(f"Throughput              : {total_queries / total_time_sec:,.0f} queries / second")
print(f"Peak VRAM Usage         : {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

Writing hardware_latency_wn18rr.py


In [21]:
!time python  hardware_latency_wn18rr.py

Running Hardware Latency Benchmark on: cuda

[1/3] Loading trained weights from /kaggle/working/best_wn18rr_final.pt...
 -> Compiling modalities into static entity representation...

[2/3] --- BENCHMARK 1: Single Query Latency (Batch Size = 1) ---
Average Latency per query : 0.577 ms
99th Percentile (p99)     : 0.758 ms

[3/3] --- BENCHMARK 2: Throughput (Batch Size = 1024) ---
Total Queries Evaluated : 102,400
Total Time              : 1.186 seconds
Throughput              : 86,352 queries / second
Peak VRAM Usage         : 1053.24 MB

real	0m6.996s
user	0m5.648s
sys	0m1.827s


Per proposed models latency test is done and a major success. To prove lets compare with current best models

hits oom at higher batch sizes. like 1024 needs 79gb of vram

In [28]:
%%writefile benchmark_baselines_latency.py

import torch
import time
import numpy as np
from pykeen.datasets import PathDataset
from pykeen.models import TransE, ComplEx, CompGCN
import glob
import gc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Benchmarking Baselines on: {DEVICE}")

# 1. Load Dataset Metadata
train_path = glob.glob("wn18rr_data/**/train.txt", recursive=True)[0]
valid_path = glob.glob("wn18rr_data/**/valid.txt", recursive=True)[0]
test_path = glob.glob("wn18rr_data/**/test.txt", recursive=True)[0]

dataset = PathDataset(
    training_path=train_path,
    validation_path=valid_path,
    testing_path=test_path,
    create_inverse_triples=True  # Required for CompGCN
)
triples_factory = dataset.training

# 2. Define Models to Benchmark
models = {
    "TransE": TransE(triples_factory=triples_factory, embedding_dim=512),
    "ComplEx": ComplEx(triples_factory=triples_factory, embedding_dim=512),
    "CompGCN": CompGCN(triples_factory=triples_factory, embedding_dim=512),
}

# 3. Benchmark Function
# CRITICAL FIX: Lowering batch_size to 64 so TransE doesn't OOM with 79GB broadcast tensors
def measure_latency_and_throughput(name, model, batch_size=64, num_queries=1000):
    model = model.to(DEVICE)
    model.eval()
    
    num_entities = triples_factory.num_entities
    num_relations = triples_factory.num_relations
    
    h_single = torch.tensor([42]).to(DEVICE)
    r_single = torch.tensor([5]).to(DEVICE)
    
    # Warmup
    for _ in range(50):
        with torch.no_grad():
            _ = model.score_t(torch.stack([h_single, r_single], dim=-1))
    torch.cuda.synchronize()
    
    # Measure Single Query Latency
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    latencies = []
    
    with torch.no_grad():
        for _ in range(num_queries):
            start_event.record()
            _ = model.score_t(torch.stack([h_single, r_single], dim=-1))
            end_event.record()
            torch.cuda.synchronize()
            latencies.append(start_event.elapsed_time(end_event))
            
    avg_latency = np.mean(latencies)
    p99_latency = np.percentile(latencies, 99)
    
    # Measure Throughput (Batch Processing)
    h_batch = torch.randint(0, num_entities, (batch_size,)).to(DEVICE)
    r_batch = torch.randint(0, num_relations, (batch_size,)).to(DEVICE)
    query_batch = torch.stack([h_batch, r_batch], dim=-1)
    
    start_time = time.time()
    num_batches = 200 # Increased batches since batch_size is lower
    with torch.no_grad():
        for _ in range(num_batches):
            _ = model.score_t(query_batch)
    torch.cuda.synchronize()
    total_time = time.time() - start_time
    throughput = (num_batches * batch_size) / total_time
    
    print(f"[{name}]")
    print(f"  Single-Query Latency (Avg): {avg_latency:.3f} ms (P99: {p99_latency:.3f} ms)")
    print(f"  Throughput: {throughput:,.0f} queries/sec")
    print("-" * 50)
    
    # Aggressive VRAM cleanup for the next model
    model = model.cpu()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2) # Give CUDA a second to release the memory pool
    
    return avg_latency, p99_latency, throughput

results = {}
for name, model in models.items():
    results[name] = measure_latency_and_throughput(name, model)

Writing benchmark_all_fb15k237.py


In [27]:
!time python benchmark_baselines_latency.py

Benchmarking Baselines on: cuda
You're trying to map triples with 212 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3134 triples were filtered out
No random seed is specified. This may lead to non-reproducible results.
No random seed is specified. This may lead to non-reproducible results.
No random seed is specified. This may lead to non-reproducible results.
[TransE]
  Single-Query Latency (Avg): 2.016 ms (P99: 2.201 ms)
  Throughput: 987 queries/sec
--------------------------------------------------
[ComplEx]
  Single-Query Latency (Avg): 1.304 ms (P99: 1.616 ms)
  Throughput: 20,346 queries/sec
--------------------------------------------------
[CompGCN]
  Single-Query Latency (Avg): 1.726 ms (P99: 2.319 ms)
  Throughput: 986 queries/sec
--------------------------------------------------

real	0m48.692s
user	0m37.320s
sys	0m6.210s


In [35]:
%%writefile benchmark_all_fb15k237.py

import os
import glob
import time
import gc
import numpy as np
import torch
import torch.nn as nn
from pykeen.datasets import PathDataset
from pykeen.models import TransE, ComplEx, CompGCN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. LOAD GRAPH DATASET & PRINT TOPOLOGY STATS
print("=" * 70)
print(f"HARDWARE BENCHMARK ENVIRONMENT: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
print("=" * 70)

train_path = glob.glob("Release/**/train.txt", recursive=True)[0]
valid_path = glob.glob("Release/**/valid.txt", recursive=True)[0]
test_path = glob.glob("Release/**/test.txt", recursive=True)[0]

dataset = PathDataset(
    training_path=train_path,
    validation_path=valid_path,
    testing_path=test_path,
    create_inverse_triples=True
)

train_tf = dataset.training
valid_tf = dataset.validation
test_tf = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations # Includes inverse relations
num_train_triples = train_tf.num_triples
num_valid_triples = valid_tf.num_triples
num_test_triples = test_tf.num_triples

print("\n--- FB15K-237 GRAPH TOPOLOGY & STRUCTURAL METRICS ---")
print(f"  * Total Entity Nodes (|V|)          : {num_entities:,}")
print(f"  * Total Relation Types (|R|)        : {num_relations} (including inverses)")
print(f"  * Training Edges (|E_train|)        : {num_train_triples:,}")
print(f"  * Validation Triples (|E_valid|)    : {num_valid_triples:,}")
print(f"  * Test Triples (|E_test|)           : {num_test_triples:,}")
print(f"  * Evaluation Search Space per Query : {num_entities:,} candidate entities")
print("=" * 70)

# 2. LOAD DEPLOYED STRUCTURALBERT MODEL
class DeployedStructuralBERT(nn.Module):
    def __init__(self, final_entity_embeddings, relation_embeddings):
        super().__init__()
        self.e_entity = nn.Embedding.from_pretrained(final_entity_embeddings, freeze=True)
        self.relation_emb = nn.Embedding.from_pretrained(relation_embeddings, freeze=True)
        
    def score_t(self, query_batch):
        h_idx = query_batch[:, 0]
        r_idx = query_batch[:, 1]
        
        h_emb = self.e_entity(h_idx)
        r_emb = self.relation_emb(r_idx)
        t_emb = self.e_entity.weight 
        
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        
        scores = torch.matmul(hr_re, t_re.t()) + torch.matmul(hr_im, t_im.t())
        return scores

class StructuralBERT_Final(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=512):
        super().__init__()
        self.struct_proj = nn.Sequential(
            nn.Linear(512, embedding_dim), nn.GELU(),
            nn.LayerNorm(embedding_dim), nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", torch.zeros(num_entities, 512))
        self.text_proj = nn.Linear(768, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, 768))
        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        
        self.glu_value = nn.Linear(1024, embedding_dim)
        self.glu_gate = nn.Linear(1024, embedding_dim)
        self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

    def _fuse(self, txt, struct):
        combined = torch.cat([txt, struct], dim=-1)
        value = self.glu_value(combined)
        gate = self.glu_gate(combined)
        fused = value * torch.nn.functional.gelu(gate)
        return self.glu_mix(fused)

    def entity_rep(self):
        ids = torch.arange(num_entities, device=self.struct_cache.device)
        txt = self.text_proj(self.bert_cache[ids])
        struct = self.struct_proj(self.struct_cache[ids])
        return self._fuse(txt, struct) + self.entity_residual(ids)

def load_structural_bert_deployed(checkpoint_path):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")
    raw_model = StructuralBERT_Final(num_entities, num_relations).to(DEVICE)
    raw_model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True), strict=False)
    raw_model.eval()
    
    with torch.no_grad():
        final_entities = raw_model.entity_rep()
        final_relations = raw_model.relation_emb.weight.clone()
        
    deployed = DeployedStructuralBERT(final_entities, final_relations).to(DEVICE)
    deployed.eval()
    del raw_model
    torch.cuda.empty_cache()
    return deployed

# 3. BENCHMARKING ENGINE
test_triples = test_tf.mapped_triples.to(DEVICE)
test_queries = test_triples[:, :2] # All (h, r) queries from the test dataset

def benchmark_model(name, model, batch_size=64):
    torch.cuda.reset_peak_memory_stats()
    model = model.to(DEVICE)
    model.eval()
    
    # 1. Warmup
    sample_query = test_queries[:1]
    for _ in range(50):
        with torch.no_grad():
            _ = model.score_t(sample_query)
    torch.cuda.synchronize()
    
    # 2. Single-Query Real-Time Latency (1,000 runs)
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    single_latencies = []
    
    with torch.no_grad():
        for i in range(1000):
            query = test_queries[i % len(test_queries) : (i % len(test_queries)) + 1]
            start_event.record()
            _ = model.score_t(query)
            end_event.record()
            torch.cuda.synchronize()
            single_latencies.append(start_event.elapsed_time(end_event))
            
    avg_latency = float(np.mean(single_latencies))
    p99_latency = float(np.percentile(single_latencies, 99))
    
    # 3. Full Test Dataset Sweep 
    t0_full = time.time()
    num_test = len(test_queries)
    with torch.no_grad():
        for start_idx in range(0, num_test, batch_size):
            batch = test_queries[start_idx : start_idx + batch_size]
            _ = model.score_t(batch)
    torch.cuda.synchronize()
    total_test_eval_time = time.time() - t0_full
    
    # 4. Throughput Calculation
    throughput = num_test / total_test_eval_time
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    
    print(f"[{name}]")
    print(f"  * Single-Query Latency (Avg) : {avg_latency:.3f} ms")
    print(f"  * Single-Query Latency (P99) : {p99_latency:.3f} ms")
    print(f"  * Full Test Set Eval Time    : {total_test_eval_time:.3f} s ({num_test:,} queries)")
    print(f"  * Inference Throughput       : {throughput:,.0f} queries/sec")
    print(f"  * Peak VRAM Usage            : {peak_vram:.2f} MB")
    print("-" * 70)
    
    model = model.cpu()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)
    
    return {
        "latency_avg": avg_latency,
        "latency_p99": p99_latency,
        "full_time": total_test_eval_time,
        "throughput": throughput,
        "vram": peak_vram
    }

# 4. RUN FULL SCREENING BENCHMARK
results = {}

print("\n--- RUNNING SYSTEM BENCHMARK ON ALL TEST DATA ---")
# Keep batch_size=64 for baselines to avoid 30GB+ VRAM tensor expansions
results["TransE"] = benchmark_model("TransE", TransE(triples_factory=train_tf, embedding_dim=512), batch_size=64)
results["ComplEx"] = benchmark_model("ComplEx", ComplEx(triples_factory=train_tf, embedding_dim=512), batch_size=64)
results["CompGCN"] = benchmark_model("CompGCN", CompGCN(triples_factory=train_tf, embedding_dim=512), batch_size=64)

checkpoint_path = "best_fb15k237_final.pt"
deployed_sb = load_structural_bert_deployed(checkpoint_path)

# CHANGED: Now StructuralBERT strictly uses batch_size=64 to match the baselines
results["StructuralBERT (Ours)"] = benchmark_model("StructuralBERT (Ours)", deployed_sb, batch_size=64)

# 5. SUMMARY COMPARISON TABLE
print("\n" + "=" * 70)
print(f"{'Model':<24} | {'Latency (ms)':<12} | {'Throughput (q/s)':<18} | {'Peak VRAM (MB)':<14}")
print("-" * 70)
for model_name, metrics in results.items():
    print(f"{model_name:<24} | {metrics['latency_avg']:<12.3f} | {metrics['throughput']:<18,.0f} | {metrics['vram']:<14.2f}")
print("=" * 70)

Overwriting benchmark_all_fb15k237.py


In [36]:
!time python  benchmark_all_fb15k237.py

HARDWARE BENCHMARK ENVIRONMENT: cuda
GPU Device Name: Tesla T4
You're trying to map triples with 30 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 28 from 20466 triples were filtered out
You're trying to map triples with 9 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 9 from 17535 triples were filtered out

--- FB15K-237 GRAPH TOPOLOGY & STRUCTURAL METRICS ---
  * Total Entity Nodes (|V|)          : 14,505
  * Total Relation Types (|R|)        : 474 (including inverses)
  * Training Edges (|E_train|)        : 272,115
  * Validation Triples (|E_valid|)    : 17,526
  * Test Triples (|E_test|)           : 20,438
  * Evaluation Search Space per Query : 14,505 candidate entities

--- RUNNING SYSTEM BENCHMARK ON ALL TEST DATA ---
No random seed is specified. This may lead to non-reproducible results.
[TransE]
  * Single-Query Latency (Avg) : 0.860 ms
  

In [33]:
%%writefile benchmark_all_wn18rr.py

import os
import glob
import time
import gc
import numpy as np
import torch
import torch.nn as nn
from pykeen.datasets import PathDataset
from pykeen.models import TransE, ComplEx, CompGCN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. LOAD GRAPH DATASET & PRINT TOPOLOGY STATS
print("=" * 70)
print(f"HARDWARE BENCHMARK ENVIRONMENT: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
print("=" * 70)

train_path = glob.glob("wn18rr_data/**/train.txt", recursive=True)[0]
valid_path = glob.glob("wn18rr_data/**/valid.txt", recursive=True)[0]
test_path = glob.glob("wn18rr_data/**/test.txt", recursive=True)[0]

dataset = PathDataset(
    training_path=train_path,
    validation_path=valid_path,
    testing_path=test_path,
    create_inverse_triples=True
)

train_tf = dataset.training
valid_tf = dataset.validation
test_tf = dataset.testing

num_entities = train_tf.num_entities
num_relations = train_tf.num_relations 
num_train_triples = train_tf.num_triples
num_valid_triples = valid_tf.num_triples
num_test_triples = test_tf.num_triples

print("\n--- WN18RR GRAPH TOPOLOGY & STRUCTURAL METRICS ---")
print(f"  * Total Entity Nodes (|V|)          : {num_entities:,}")
print(f"  * Total Relation Types (|R|)        : {num_relations} (including inverses)")
print(f"  * Training Edges (|E_train|)        : {num_train_triples:,}")
print(f"  * Validation Triples (|E_valid|)    : {num_valid_triples:,}")
print(f"  * Test Triples (|E_test|)           : {num_test_triples:,}")
print(f"  * Evaluation Search Space per Query : {num_entities:,} candidate entities")
print("=" * 70)

# 2. LOAD DEPLOYED STRUCTURALBERT MODEL
class DeployedStructuralBERT(nn.Module):
    def __init__(self, final_entity_embeddings, relation_embeddings):
        super().__init__()
        self.e_entity = nn.Embedding.from_pretrained(final_entity_embeddings, freeze=True)
        self.relation_emb = nn.Embedding.from_pretrained(relation_embeddings, freeze=True)
        
    def score_t(self, query_batch):
        h_idx = query_batch[:, 0]
        r_idx = query_batch[:, 1]
        
        h_emb = self.e_entity(h_idx)
        r_emb = self.relation_emb(r_idx)
        t_emb = self.e_entity.weight 
        
        h_re, h_im = torch.chunk(h_emb, 2, dim=-1)
        r_re, r_im = torch.chunk(r_emb, 2, dim=-1)
        t_re, t_im = torch.chunk(t_emb, 2, dim=-1)
        
        hr_re = h_re * r_re - h_im * r_im
        hr_im = h_re * r_im + h_im * r_re
        
        scores = torch.matmul(hr_re, t_re.t()) + torch.matmul(hr_im, t_im.t())
        return scores

class StructuralBERT_Final(nn.Module):
    def __init__(self, num_entities, num_relations, embedding_dim=512):
        super().__init__()
        self.struct_proj = nn.Sequential(
            nn.Linear(512, embedding_dim), nn.GELU(),
            nn.LayerNorm(embedding_dim), nn.Linear(embedding_dim, embedding_dim)
        )
        self.register_buffer("struct_cache", torch.zeros(num_entities, 512))
        self.text_proj = nn.Linear(768, embedding_dim)
        self.register_buffer("bert_cache", torch.zeros(num_entities, 768))
        self.entity_residual = nn.Embedding(num_entities, embedding_dim)
        self.relation_emb = nn.Embedding(num_relations, embedding_dim)
        
        self.glu_value = nn.Linear(1024, embedding_dim)
        self.glu_gate = nn.Linear(1024, embedding_dim)
        self.glu_mix = nn.Linear(embedding_dim, embedding_dim)

    def _fuse(self, txt, struct):
        combined = torch.cat([txt, struct], dim=-1)
        value = self.glu_value(combined)
        gate = self.glu_gate(combined)
        fused = value * torch.nn.functional.gelu(gate)
        return self.glu_mix(fused)

    def entity_rep(self):
        ids = torch.arange(num_entities, device=self.struct_cache.device)
        txt = self.text_proj(self.bert_cache[ids])
        struct = self.struct_proj(self.struct_cache[ids])
        return self._fuse(txt, struct) + self.entity_residual(ids)

def load_structural_bert_deployed(checkpoint_path):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")
    raw_model = StructuralBERT_Final(num_entities, num_relations).to(DEVICE)
    raw_model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True), strict=False)
    raw_model.eval()
    
    with torch.no_grad():
        final_entities = raw_model.entity_rep()
        final_relations = raw_model.relation_emb.weight.clone()
        
    deployed = DeployedStructuralBERT(final_entities, final_relations).to(DEVICE)
    deployed.eval()
    del raw_model
    torch.cuda.empty_cache()
    return deployed

# 3. BENCHMARKING ENGINE
test_triples = test_tf.mapped_triples.to(DEVICE)
test_queries = test_triples[:, :2] 

def benchmark_model(name, model, batch_size=64):
    torch.cuda.reset_peak_memory_stats()
    model = model.to(DEVICE)
    model.eval()
    
    sample_query = test_queries[:1]
    for _ in range(50):
        with torch.no_grad():
            _ = model.score_t(sample_query)
    torch.cuda.synchronize()
    
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    single_latencies = []
    
    with torch.no_grad():
        for i in range(1000):
            query = test_queries[i % len(test_queries) : (i % len(test_queries)) + 1]
            start_event.record()
            _ = model.score_t(query)
            end_event.record()
            torch.cuda.synchronize()
            single_latencies.append(start_event.elapsed_time(end_event))
            
    avg_latency = float(np.mean(single_latencies))
    p99_latency = float(np.percentile(single_latencies, 99))
    
    t0_full = time.time()
    num_test = len(test_queries)
    with torch.no_grad():
        for start_idx in range(0, num_test, batch_size):
            batch = test_queries[start_idx : start_idx + batch_size]
            _ = model.score_t(batch)
    torch.cuda.synchronize()
    total_test_eval_time = time.time() - t0_full
    
    throughput = num_test / total_test_eval_time
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    
    print(f"[{name}]")
    print(f"  * Single-Query Latency (Avg) : {avg_latency:.3f} ms")
    print(f"  * Single-Query Latency (P99) : {p99_latency:.3f} ms")
    print(f"  * Full Test Set Eval Time    : {total_test_eval_time:.3f} s ({num_test:,} queries)")
    print(f"  * Inference Throughput       : {throughput:,.0f} queries/sec")
    print(f"  * Peak VRAM Usage            : {peak_vram:.2f} MB")
    print("-" * 70)
    
    model = model.cpu()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(2)
    
    return {
        "latency_avg": avg_latency,
        "latency_p99": p99_latency,
        "full_time": total_test_eval_time,
        "throughput": throughput,
        "vram": peak_vram
    }

# 4. RUN FULL SCREENING BENCHMARK
results = {}

print("\n--- RUNNING SYSTEM BENCHMARK ON ALL TEST DATA ---")
results["TransE"] = benchmark_model("TransE", TransE(triples_factory=train_tf, embedding_dim=512), batch_size=64)
results["ComplEx"] = benchmark_model("ComplEx", ComplEx(triples_factory=train_tf, embedding_dim=512), batch_size=64)
results["CompGCN"] = benchmark_model("CompGCN", CompGCN(triples_factory=train_tf, embedding_dim=512), batch_size=64)

checkpoint_path = "/kaggle/working/best_wn18rr_final.pt" 
deployed_sb = load_structural_bert_deployed(checkpoint_path)

# CHANGED: Now StructuralBERT strictly uses batch_size=64 to match the baselines
results["StructuralBERT (Ours)"] = benchmark_model("StructuralBERT (Ours)", deployed_sb, batch_size=64)

# 5. SUMMARY COMPARISON TABLE
print("\n" + "=" * 70)
print(f"{'Model':<24} | {'Latency (ms)':<12} | {'Throughput (q/s)':<18} | {'Peak VRAM (MB)':<14}")
print("-" * 70)
for model_name, metrics in results.items():
    print(f"{model_name:<24} | {metrics['latency_avg']:<12.3f} | {metrics['throughput']:<18,.0f} | {metrics['vram']:<14.2f}")
print("=" * 70)

Overwriting benchmark_all_wn18rr.py


In [34]:
!time python benchmark_all_wn18rr.py

HARDWARE BENCHMARK ENVIRONMENT: cuda
GPU Device Name: Tesla T4
You're trying to map triples with 212 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3134 triples were filtered out
You're trying to map triples with 211 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3034 triples were filtered out

--- WN18RR GRAPH TOPOLOGY & STRUCTURAL METRICS ---
  * Total Entity Nodes (|V|)          : 40,559
  * Total Relation Types (|R|)        : 22 (including inverses)
  * Training Edges (|E_train|)        : 86,835
  * Validation Triples (|E_valid|)    : 2,824
  * Test Triples (|E_test|)           : 2,924
  * Evaluation Search Space per Query : 40,559 candidate entities

--- RUNNING SYSTEM BENCHMARK ON ALL TEST DATA ---
No random seed is specified. This may lead to non-reproducible results.
[TransE]
  * Single-Query Latency (Avg) : 1.970 ms
  * S